## Nuscenes mini dataset preparation and setup

In [ ]:
!mkdir -p /data/sets/nuscenes  # Make the directory to store the nuScenes dataset in.

In [ ]:
!wget https://www.nuscenes.org/data/v1.0-mini.tgz  # Download the nuScenes mini split.

In [ ]:
!tar -xf v1.0-mini.tgz -C /data/sets/nuscenes  # Uncompress the nuScenes mini split.

### Install Compatible Dependencies
Downgrade numpy/scipy/scikit-learn to versions compatible with nuscenes-devkit.

In [ ]:
!pip uninstall -y numpy scipy scikit-learn
!pip install numpy==1.26.4 scipy==1.11.4 scikit-learn==1.3.2        # Install compatible numpy, scipy, and scikit-learn

### Restart Runtime
**Required after pip downgrades** — Colab must reload the environment for version changes to take effect.

In [ ]:
import os
os.kill(os.getpid(), 9)     # Restart runtime

### Install NuScenes DevKit and Ultralytics

In [ ]:
!pip install nuscenes-devkit        # Install nuscenes-devkit

In [ ]:
!pip install nuscenes-devkit ultralytics -q     # Install nuScenes-devkit ultralytics

### Verify Imports

In [ ]:
from nuscenes.nuscenes import NuScenes
from ultralytics import YOLO
import numpy as np
print("✅ nuscenes-devkit OK")
print("✅ ultralytics OK")

### Mount Google Drive and Validate Dataset Structure

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

NUSCENES_ROOT = '/data/sets/nuscenes'

import os
expected = ['samples', 'sweeps', 'maps', 'v1.0-mini']
for folder in expected:
  path = os.path.join(NUSCENES_ROOT, folder)
  status = "✅" if os.path.exists(path) else "missing"
  print(f"{status} {folder}/")

### Visualize NuScenes Sample and Test YOLO Detection
Sanity check — renders a CAM_FRONT scene and runs YOLO26n inference to confirm the pipeline works end-to-end before training.

In [ ]:
nusc = NuScenes(version='v1.0-mini', dataroot=NUSCENES_ROOT, verbose=True)
my_sammple = nusc.sample[0]
nusc.render_sample(my_sammple['token'])         # NuScenes render_sample

### Run YOLO Detection on CAM_FRONT Image

In [ ]:
from PIL import Image
cam_token = my_sample['data']['CAM_FRONT']
cam_data = nusc.get('sample_data', cam_token)
img_path = os.path.join(NUSCENES_ROOT, cam_data['filename'])
print(f"Image path: {img_path}")
img = Image.open(img_path)
model = YOLO("yolo26n.pt")
results = model(img_path)
results[0].show()
print(f"✅ YOLO26 detected {len(results[0].boxes)} objects")        # yolo26n.pt model object detection on NuScenes sample 'CAM_FRONT' image test

### Set Output Directory

In [ ]:
OUTPUT_DIR = '/content/nuscenes_yolo'

## YOLO26-NuScenes Detection Conversion and Training

### Create Detection YAML Config

In [ ]:
# Create yaml file
yaml_content = f"""
path: {OUTPUT_DIR}
train: images/train
val: images/val

names:
  0: car
  1: pedestrian
  2: bicycle
  3: motorcycle
  4: bus
  5: truck
  6: traffic_cone
  7: barrier
"""

yaml_path = f'{OUTPUT_DIR}/nuscenes.yaml'
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print(open(yaml_path).read())
print(f"✅ Saved to {yaml_path}")

### Convert NuScenes Annotations to YOLO Detection Format
Projects 3D bounding boxes from the global frame into the CAM_FRONT image plane and converts them to normalized YOLO format (cx, cy, w, h). Uses an 80/20 train/val split with seed=42.

In [ ]:
import os
import numpy as np
from nuscenes.nuscenes import NuScenes
from nuscenes.utils.geometry_utils import view_points, BoxVisibility
from nuscenes.utils.data_classes import Box
from pyquaternion import Quaternion
from pathlib import Path
import shutil
import random

NUSCENES_ROOT = '/data/sets/nuscenes'
OUTPUT_DIR = '/content/nuscenes_yolo'

CLASS_MAP = {
    'car': 0, 'pedestrian': 1, 'bicycle': 2, 'motorcycle': 3,
    'bus': 4, 'truck': 5, 'traffic_cone': 6, 'barrier': 7,
}

nusc = NuScenes(version='v1.0-mini', dataroot=NUSCENES_ROOT, verbose=False)

# Fresh output folders
if Path(OUTPUT_DIR).exists():
    shutil.rmtree(OUTPUT_DIR)
for split in ['train', 'val']:
    Path(f'{OUTPUT_DIR}/images/{split}').mkdir(parents=True, exist_ok=True)
    Path(f'{OUTPUT_DIR}/labels/{split}').mkdir(parents=True, exist_ok=True)

# Collect all samples
all_samples = []
for scene in nusc.scene:
    token = scene['first_sample_token']
    while token:
        all_samples.append(nusc.get('sample', token))
        token = nusc.get('sample', token)['next']

random.seed(42)
random.shuffle(all_samples)
split_idx = int(len(all_samples) * 0.8)
splits = {'train': all_samples[:split_idx], 'val': all_samples[split_idx:]}
print(f"Total: {len(all_samples)} → train: {split_idx}, val: {len(all_samples)-split_idx}")

converted, total_boxes = 0, 0

for split, samples in splits.items():
    for sample in samples:
        cam_token = sample['data']['CAM_FRONT']
        cam_data  = nusc.get('sample_data', cam_token)
        img_path  = os.path.join(NUSCENES_ROOT, cam_data['filename'])
        img_w, img_h = cam_data['width'], cam_data['height']

        # Camera calibration
        cs  = nusc.get('calibrated_sensor', cam_data['calibrated_sensor_token'])
        ep  = nusc.get('ego_pose', cam_data['ego_pose_token'])
        intrinsic = np.array(cs['camera_intrinsic'])

        # Get boxes in global frame, transform to camera frame
        boxes = nusc.get_boxes(cam_token)
        labels = []

        for box in boxes:
            # Map class
            cat_simple = next((k for k in CLASS_MAP if k in box.name), None)
            if cat_simple is None:
                continue

            # 1. Global → ego vehicle frame
            box.translate(-np.array(ep['translation']))
            box.rotate(Quaternion(ep['rotation']).inverse)

            # 2. Ego → camera frame
            box.translate(-np.array(cs['translation']))
            box.rotate(Quaternion(cs['rotation']).inverse)

            # 3. Skip if behind camera (z < 0.1 means too close or behind)
            if box.center[2] < 0.1:
                continue

            # 4. Project 3D corners to 2D image plane
            corners = view_points(box.corners(), intrinsic, normalize=True)
            xs, ys = corners[0], corners[1]

            x1, x2 = xs.min(), xs.max()
            y1, y2 = ys.min(), ys.max()

            # 5. Skip if completely outside image
            if x2 < 0 or x1 > img_w or y2 < 0 or y1 > img_h:
                continue

            # 6. Clip to image bounds
            x1 = np.clip(x1, 0, img_w)
            x2 = np.clip(x2, 0, img_w)
            y1 = np.clip(y1, 0, img_h)
            y2 = np.clip(y2, 0, img_h)

            # 7. Skip tiny or degenerate boxes
            if x2 - x1 < 4 or y2 - y1 < 4:
                continue
            if (x2 - x1) / img_w > 0.95 or (y2 - y1) / img_h > 0.95:
                continue

            # 8. YOLO format (normalized center + size)
            cx = (x1 + x2) / 2 / img_w
            cy = (y1 + y2) / 2 / img_h
            w  = (x2 - x1) / img_w
            h  = (y2 - y1) / img_h

            labels.append(f"{CLASS_MAP[cat_simple]} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
            total_boxes += 1

        # Save symlink + label
        img_name = Path(img_path).name
        dst = f'{OUTPUT_DIR}/images/{split}/{img_name}'
        if not os.path.exists(dst):
            os.symlink(img_path, dst)

        with open(f'{OUTPUT_DIR}/labels/{split}/{Path(img_name).stem}.txt', 'w') as f:
            f.write('\n'.join(labels))

        converted += 1

print(f"✅ Done: {converted} images, {total_boxes} total boxes")

# Verify
for split in ['train', 'val']:
    files = list(Path(f'{OUTPUT_DIR}/labels/{split}').glob('*.txt'))
    non_empty = sum(1 for f in files if f.stat().st_size > 0)
    print(f"  {split}: {len(files)} files, {non_empty} non-empty")

# Spot check — should see reasonable values (not 1.0)
from pathlib import Path
sample = next(f for f in Path(f'{OUTPUT_DIR}/labels/train').glob('*.txt') if f.stat().st_size > 0)
print(f"\nSample label ({sample.name}):")
print(open(sample).read()[:300])

### Set Drive Save Path
Point output and run directories to Google Drive so training checkpoints persist across Colab sessions.

In [ ]:
OUTPUT_DIR = '/content/drive/MyDrive/nuscenes_yolo'
RUNS_DIR = '/content/drive/MyDrive/yolo_runs'       # Set MyDrive as trained model save path

### Train YOLO26n Detection Model

In [ ]:
# Train YOLO26 Detection Model
from ultralytics import YOLO

model = YOLO("yolo26n.pt")

model.train(
    data=yaml_path,
    epochs=100,
    imgsz=640,
    batch=16,
    name="yolo26n_nuscenes",
    project=RUNS_DIR,
    device=0,
    workers=2,
    patience=20,
    save=True,
    plots=True,
)

## YOLO26-NuScenes Segmentation Conversion and Training

### Convert NuScenes to YOLO Segmentation Format
Projects 3D box corners into the image plane and computes a convex hull polygon for each object — gives YOLO-seg the full silhouette outline instead of just a bounding box.

In [ ]:
import os
import numpy as np
import shutil
from pathlib import Path
from scipy.spatial import ConvexHull
from pyquaternion import Quaternion
from nuscenes.nuscenes import NuScenes
from nuscenes.utils.geometry_utils import view_points
import random

NUSCENES_ROOT = '/data/sets/nuscenes'
SEG_DIR       = '/content/drive/MyDrive/yolo_runs/nuscenes_yolo_seg'   # Save data to MyDrive

CLASS_MAP = {
    'car': 0, 'pedestrian': 1, 'bicycle': 2, 'motorcycle': 3,
    'bus': 4, 'truck': 5, 'traffic_cone': 6, 'barrier': 7,
}

# Min polygon points and min area thresholds
MIN_POLYGON_PTS = 4
MIN_AREA_RATIO  = 0.0001  # at least 0.01% of image area
MAX_AREA_RATIO  = 0.90    # at most 90% of image area

nusc = NuScenes(version='v1.0-mini', dataroot=NUSCENES_ROOT, verbose=False)

# Fresh output directories
if Path(SEG_DIR).exists():
    shutil.rmtree(SEG_DIR)
for split in ['train', 'val']:
    Path(f'{SEG_DIR}/images/{split}').mkdir(parents=True, exist_ok=True)
    Path(f'{SEG_DIR}/labels/{split}').mkdir(parents=True, exist_ok=True)

# Collect all samples and split 80/20
all_samples = []
for scene in nusc.scene:
    token = scene['first_sample_token']
    while token:
        all_samples.append(nusc.get('sample', token))
        token = nusc.get('sample', token)['next']

random.seed(42)
random.shuffle(all_samples)
split_idx = int(len(all_samples) * 0.8)
splits = {'train': all_samples[:split_idx], 'val': all_samples[split_idx:]}
print(f"Total: {len(all_samples)} → train: {split_idx}, val: {len(all_samples) - split_idx}")

def project_box_to_polygon(box, intrinsic, img_w, img_h):
    """
    Project 3D box corners to image plane, return convex hull polygon
    in YOLO normalized format, or None if invalid.
    """
    # Get all 8 corners of the 3D box
    corners_3d = box.corners()  # shape (3, 8)

    # Project to image plane
    corners_2d = view_points(corners_3d, intrinsic, normalize=True)  # (3, 8)

    # Check all corners are in front of camera
    if any(corners_2d[2] < 0):
        return None

    xs = corners_2d[0]
    ys = corners_2d[1]

    # Filter corners that fall within or near the image
    valid = (xs >= -img_w * 0.1) & (xs <= img_w * 1.1) & \
            (ys >= -img_h * 0.1) & (ys <= img_h * 1.1)

    if valid.sum() < MIN_POLYGON_PTS:
        return None

    xs_valid = xs[valid]
    ys_valid = ys[valid]

    # Compute convex hull of valid projected corners
    pts = np.stack([xs_valid, ys_valid], axis=1)

    try:
        if len(pts) >= 3:
            hull = ConvexHull(pts)
            hull_pts = pts[hull.vertices]
        else:
            hull_pts = pts
    except Exception:
        # Degenerate case (collinear points) — use bounding rect instead
        x1, x2 = xs_valid.min(), xs_valid.max()
        y1, y2 = ys_valid.min(), ys_valid.max()
        hull_pts = np.array([[x1,y1],[x2,y1],[x2,y2],[x1,y2]])

    # Clip polygon points to image bounds
    hull_pts[:, 0] = np.clip(hull_pts[:, 0], 0, img_w)
    hull_pts[:, 1] = np.clip(hull_pts[:, 1], 0, img_h)

    # Check polygon area is reasonable
    w_span = hull_pts[:, 0].max() - hull_pts[:, 0].min()
    h_span = hull_pts[:, 1].max() - hull_pts[:, 1].min()
    area_ratio = (w_span * h_span) / (img_w * img_h)

    if area_ratio < MIN_AREA_RATIO or area_ratio > MAX_AREA_RATIO:
        return None

    # Normalize to [0, 1]
    hull_pts[:, 0] /= img_w
    hull_pts[:, 1] /= img_h

    return hull_pts


# Convert all samples
converted = 0
total_polygons = 0
skipped_behind = 0
skipped_class = 0
skipped_area = 0

for split, samples in splits.items():
    for sample in samples:
        cam_token = sample['data']['CAM_FRONT']
        cam_data  = nusc.get('sample_data', cam_token)
        img_path  = os.path.join(NUSCENES_ROOT, cam_data['filename'])
        img_w, img_h = cam_data['width'], cam_data['height']

        # Camera calibration
        cs = nusc.get('calibrated_sensor', cam_data['calibrated_sensor_token'])
        ep = nusc.get('ego_pose', cam_data['ego_pose_token'])
        intrinsic = np.array(cs['camera_intrinsic'])

        boxes = nusc.get_boxes(cam_token)
        labels = []

        for box in boxes:
            # Map to our classes
            cat_simple = next((k for k in CLASS_MAP if k in box.name), None)
            if cat_simple is None:
                skipped_class += 1
                continue

            # Transform: global → ego → camera frame
            box.translate(-np.array(ep['translation']))
            box.rotate(Quaternion(ep['rotation']).inverse)
            box.translate(-np.array(cs['translation']))
            box.rotate(Quaternion(cs['rotation']).inverse)

            # Skip objects behind camera
            if box.center[2] < 0.1:
                skipped_behind += 1
                continue

            # Project to polygon
            polygon = project_box_to_polygon(box, intrinsic, img_w, img_h)
            if polygon is None:
                skipped_area += 1
                continue

            # Format: class x1 y1 x2 y2 x3 y3 ... (YOLO seg format)
            coords = ' '.join([f"{x:.6f} {y:.6f}" for x, y in polygon])
            labels.append(f"{CLASS_MAP[cat_simple]} {coords}")
            total_polygons += 1

        # Symlink image
        img_name = Path(img_path).name
        dst_img = f'{SEG_DIR}/images/{split}/{img_name}'
        if not os.path.exists(dst_img):
            os.symlink(img_path, dst_img)

        # Write label
        with open(f'{SEG_DIR}/labels/{split}/{Path(img_name).stem}.txt', 'w') as f:
            f.write('\n'.join(labels))

        converted += 1

print(f"\n✅ Done: {converted} images, {total_polygons} polygons")
print(f"   Skipped → behind camera: {skipped_behind}, bad area: {skipped_area}, unknown class: {skipped_class}")

# Verify quality
print("\n--- Quality check ---")
for split in ['train', 'val']:
    files  = list(Path(f'{SEG_DIR}/labels/{split}').glob('*.txt'))
    filled = sum(1 for f in files if f.stat().st_size > 0)
    print(f"  {split}: {len(files)} files, {filled} non-empty")

# Show a sample polygon
sample_f = next(f for f in Path(f'{SEG_DIR}/labels/train').glob('*.txt') if f.stat().st_size > 0)
first_line = open(sample_f).readline().strip().split()
cls_id = first_line[0]
n_points = (len(first_line) - 1) // 2
print(f"\nSample: class={cls_id}, polygon has {n_points} points (rectangle=4, convex hull=4–8)")
print(' '.join(first_line[:9]), '...')

### Create Segmentation YAML Config

In [ ]:
# Create yaml file
seg_yaml = f"""
path: {SEG_DIR}
train: images/train
val: images/val

names:
  0: car
  1: pedestrian
  2: bicycle
  3: motorcycle
  4: bus
  5: truck
  6: traffic_cone
  7: barrier
"""

seg_yaml_path = f'{SEG_DIR}/nuscenes_seg.yaml'
with open(seg_yaml_path, 'w') as f:
    f.write(seg_yaml)
print(f"✅ YAML saved → {seg_yaml_path}")

### Train YOLO26n Segmentation Model

In [ ]:
from ultralytics import YOLO

seg_yaml_path = '/content/drive/MyDrive/yolo_runs/nuscenes_yolo_seg/nuscenes_seg.yaml'

model = YOLO("yolo26n-seg.pt")
model.train(
    data=seg_yaml_path,
    epochs=100,
    imgsz=640,
    batch=16,
    name="yolo26n_seg_nuscenes",
    project=RUNS_DIR,
    device=0,
    workers=2,
    patience=20,
    save=True,
    plots=True,
)

## YOLOv8-NuScenes Detection Conversion and Training

In [ ]:
from ultralytics import YOLO

yaml_path = '/content/drive/MyDrive/yolo_runs/nuscenes_yolo_seg/nuscenes_seg.yaml'
model = YOLO("yolov8n.pt")

model.train(
    data=yaml_path,
    epochs=100,
    imgsz=640,
    batch=16,
    name="yolov8n_nuscenes",
    project=RUNS_DIR,
    device=0,
    workers=2,
    patience=20,
    save=True,
    plots=True,
)

## Nuscenes Video Export

In [ ]:
import cv2
import os
from nuscenes.nuscenes import NuScenes
from pathlib import Path

NUSCENES_ROOT = '/data/sets/nuscenes'
VIDEO_DIR = '/content/videos'
Path(VIDEO_DIR).mkdir(exist_ok=True)

nusc = NuScenes(version='v1.0-mini', dataroot=NUSCENES_ROOT, verbose=False)

for i, scene in enumerate(nusc.scene):
    out_path = f'{VIDEO_DIR}/scene_{i:02d}_{scene["name"]}.mp4'

    # Start from first SAMPLE (keyframe)
    sample_token = scene['first_sample_token']
    sample = nusc.get('sample', sample_token)

    # Get first cam_front sample_data token
    cam_sd_token = sample['data']['CAM_FRONT']

    # Collect ALL sample_data tokens (keyframes + sweeps) in order
    all_frames = []
    current_token = cam_sd_token
    while current_token:
        sd = nusc.get('sample_data', current_token)
        all_frames.append(sd)
        current_token = sd['next']

    # Write video
    writer = None
    for sd in all_frames:
        img_path = os.path.join(NUSCENES_ROOT, sd['filename'])
        if not os.path.exists(img_path):
            continue
        frame = cv2.imread(img_path)
        if frame is None:
            continue
        if writer is None:
            h, w = frame.shape[:2]
            writer = cv2.VideoWriter(
                out_path,
                cv2.VideoWriter_fourcc(*'mp4v'),
                12, (w, h)
            )
        writer.write(frame)

    if writer:
        writer.release()
    print(f"✅ Scene {i:02d}: {len(all_frames)} frames → {Path(out_path).name}")

print(f"\nAll videos saved to {VIDEO_DIR}")

## Ultralytics Solution Demo on Trained YOLO26 Model

In [ ]:
import cv2
import os
from ultralytics import solutions
from pthlib import Path

MODEL = '/content/drive/MyDrive/yolo_runs/yolo26n_nuscenes/weights/best.pt'
VIDEO_DIR = '/content/videos'
videos = sorted(os.listdir(VIDEO_DIR))
VIDEO = f'{VIDEO_DIR}/{videos[2]}'
OUTPUT_DIR = '/content/solutions_output'
Path(OUTPUT_DIR).mkdir(exist_ok=True)

def get_video_props(path):
    cap = cv2.VideoCapture(path)
    w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 12
    cap.release()
    return w, h, fps

W, H, FPS = get_video_props(VIDEO)
print(f"Video: {VIDEO}")
print(f"Size: {W}x{H} @ {FPS:.1f} FPS")

# ── 1. Speed Estimation ──────────────────────────────────────────
print("\nRunning Speed Estimator...")
cap = cv2.VideoCapture(VIDEO)
speed_obj = solutions.SpeedEstimator(model=MODEL, show=False)
out = cv2.VideoWriter(f'{OUTPUT_DIR}/speed.mp4',
                      cv2.VideoWriter_fourcc(*'mp4v'), FPS, (W, H))
while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    out.write(speed_obj.process(frame).plot_im)
cap.release(); out.release()
print("✅ Speed estimation done")

# ── 2. Heatmap ───────────────────────────────────────────────────
print("Running Heatmap...")
cap = cv2.VideoCapture(VIDEO)
heat_obj = solutions.Heatmap(
    model=MODEL,
    colormap=cv2.COLORMAP_PARULA,
    show=False,
)
out = cv2.VideoWriter(f'{OUTPUT_DIR}/heatmap.mp4',
                      cv2.VideoWriter_fourcc(*'mp4v'), FPS, (W, H))
while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    out.write(heat_obj.process(frame).plot_im)
cap.release(); out.release()
print("✅ Heatmap done")

# ── 3. Object Counter ────────────────────────────────────────────
print("Running Object Counter...")
cap = cv2.VideoCapture(VIDEO)
count_obj = solutions.ObjectCounter(
    model=MODEL,
    region=[(0, int(H*0.65)), (W, int(H*0.65))],
    show=False,
)
out = cv2.VideoWriter(f'{OUTPUT_DIR}/counting.mp4',
                      cv2.VideoWriter_fourcc(*'mp4v'), FPS, (W, H))
while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    out.write(count_obj.process(frame).plot_im)
cap.release(); out.release()
print("✅ Object counting done")

# ── 4. Analytics ─────────────────────────────────────────────────
print("Running Analytics...")
cap = cv2.VideoCapture(VIDEO)
analytics_obj = solutions.Analytics(
    model=MODEL,
    analytics_type="line",
    show=False,
)
out = cv2.VideoWriter(f'{OUTPUT_DIR}/analytics.mp4',
                      cv2.VideoWriter_fourcc(*'mp4v'), FPS, (W, H))
frame_idx = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    frame_idx += 1
    out.write(analytics_obj.process(frame, frame_idx).plot_im)
cap.release(); out.release()
print("✅ Analytics done")

# ── 5. Segmentation masks ────────────────────────────────────────
print("Running Segmentation masks...")
from ultralytics import YOLO
MODEL_SEG = '/content/drive/MyDrive/yolo_runs/yolo26n_seg_nuscenes/weights/best.pt'
seg_model = YOLO(MODEL_SEG)
cap = cv2.VideoCapture(VIDEO)
out = cv2.VideoWriter(f'{OUTPUT_DIR}/segmentation_masks.mp4',
                      cv2.VideoWriter_fourcc(*'mp4v'), FPS, (W, H))
while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    out.write(seg_model(frame, verbose=False)[0].plot())
cap.release(); out.release()
print("✅ Segmentation masks done")

print(f"\nAll outputs saved to {OUTPUT_DIR}")

### Calibrated Speed Estimation
Re-runs the speed estimator with `meter_per_pixel=0.0040` derived from nuScenes camera intrinsics at ~10m depth, and excludes static objects (traffic cones, barriers).

In [ ]:
import cv2
import os
from ultralytics import solutions
from pathlib import Path

MODEL = '/content/drive/MyDrive/yolo_runs/yolo26n_nuscenes/weights/best.pt'
VIDEO_DIR = '/content/videos'
videos = sorted(os.listdir(VIDEO_DIR))
VIDEO = f'{VIDEO_DIR}/{videos[2]}'  # scene 2, stopped at traffic light
OUTPUT_DIR = '/content/solutions_output'
Path(OUTPUT_DIR).mkdir(exist_ok=True)

W = int(cv2.VideoCapture(VIDEO).get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cv2.VideoCapture(VIDEO).get(cv2.CAP_PROP_FRAME_HEIGHT))
FPS = cv2.VideoCapture(VIDEO).get(cv2.CAP_PROP_FPS) or 12

print(f"Using: {videos[2]}")
print(f"meter_per_pixel = 0.0040 (calibrated from nuScenes intrinsics at ~10m depth)")

print("\nRunning Speed Estimator (calibrated)...")
cap = cv2.VideoCapture(VIDEO)
speed_obj = solutions.SpeedEstimator(
    model=MODEL,
    show=False,
    meter_per_pixel=0.0040,
    classes=[0, 1, 2, 3, 4, 5],  # car, pedestrian, bicycle, motorcycle, bus, truck
                                   # excludes: 6=traffic_cone, 7=barrier
)
out = cv2.VideoWriter(f'{OUTPUT_DIR}/speed_calibrated.mp4',
                      cv2.VideoWriter_fourcc(*'mp4v'), FPS, (W, H))
while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    out.write(speed_obj.process(frame).plot_im)
cap.release(); out.release()

# Preview middle frame
cap = cv2.VideoCapture(f'{OUTPUT_DIR}/speed_calibrated.mp4')
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.set(cv2.CAP_PROP_POS_FRAMES, total // 2)
ret, frame = cap.read()
cap.release()
cv2.imwrite('/content/speed_preview.jpg', frame)

from IPython.display import Image, display
display(Image('/content/speed_preview.jpg'))
print(f"✅ Done — {total} frames")

## Ultralytics Segmentation Mask Demo using pre-trained COCO Weights YOLO26 Model

In [ ]:
from ultralytics import YOLO
import cv2
from pathlib import Path
import os

# Use base COCO weights, not fine-tuned
model = YOLO("yolo26n-seg.pt")

VIDEO_DIR = '/content/videos'
VIDEO = f'{VIDEO_DIR}/{sorted(os.listdir(VIDEO_DIR))[2]}'
OUTPUT_DIR = '/content/solutions_output'

cap = cv2.VideoCapture(VIDEO)
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
FPS = cap.get(cv2.CAP_PROP_FPS) or 12

out = cv2.VideoWriter(f'{OUTPUT_DIR}/segmentation_coco_cone.mp4',
                      cv2.VideoWriter_fourcc(*'mp4v'), FPS, (W, H))

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    out.write(model(frame, verbose=False)[0].plot())

cap.release(); out.release()

# Preview
cap = cv2.VideoCapture(f'{OUTPUT_DIR}/segmentation_coco_cone.mp4')
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.set(cv2.CAP_PROP_POS_FRAMES, total // 2)
ret, frame = cap.read()
cap.release()
cv2.imwrite('/content/seg_coco_preview_cone.jpg', frame)

from IPython.display import Image, display
display(Image('/content/seg_coco_preview_cone.jpg'))
print("✅ Done")

## MMDetection3d Installation and Setup for PointPillars 3D

### Step 1: Install PyTorch (CUDA 12.1)
Pin PyTorch to 2.1.0 — required for mmcv/mmdet3d compatibility.

In [ ]:
!pip install torch==2.1.0 torchvision==0.16.0 torchaudio==2.1.0 --index-url https://download.pytorch.org/whl/cu121

### Step 2: Install MMDetection3D Stack

In [ ]:
!pip install -U openmim
!mim install mmengine
!mim install mmcv==2.1.0
!mim install "mmdet>=3.0.0,<3.3.0" --quiet
!mim install "mmdet3d>=1.1.0" --quiet
!pip install nuscenes-devkit open3d --quiet

### Step 3: Verify Installation

In [ ]:
import torch
import mmcv
import mmdet
import mmdet3d
import mmengine

print(f"✅ PyTorch : {torch.__version__}")
print(f"✅ mmcv    : {mmcv.__version__}")
print(f"✅ mmdet   : {mmdet.__version__}")
print(f"✅ mmdet3d : {mmdet3d.__version__}")
print(f"✅ mmengine: {mmengine.__version__}")

### Step 4: Download Pre-trained PointPillars Weights
Official OpenMMLab checkpoint pre-trained on full nuScenes (not mini). Using this for inference/eval without fine-tuning.

In [ ]:
import os
from pathlib import Path

WEIGHTS_DIR = '/content/pointpillars_weights'
Path(WEIGHTS_DIR).mkdir(exist_ok=True)

# Official PointPillars checkpoint pretrained on nuScenes
CKPT_URL = 'https://download.openmmlab.com/mmdetection3d/v1.0.0_models/pointpillars/hv_pointpillars_fpn_sbn-all_4x8_2x_nus-3d/hv_pointpillars_fpn_sbn-all_4x8_2x_nus-3d_20210826_104936-fca299c1.pth'

!wget -q --show-progress -O {WEIGHTS_DIR}/pointpillars_nuscenes.pth {CKPT_URL}
print(f"✅ Weights saved")

### Step 5: Clone MMDetection3D Repository
Needed for the config files that define the PointPillars model architecture and data pipeline.

In [ ]:
if not os.path.exists('/content/mmdetection3d'):
    !git clone https://github.com/open-mmlab/mmdetection3d.git \
        /content/mmdetection3d --depth 1 --quiet
    print("✅ Cloned")
else:
    print("✅ Already exists")

# Verify config exists
cfg = '/content/mmdetection3d/configs/pointpillars/hv_pointpillars_fpn_sbn-all_4x8_2x_nus-3d.py'
print(f"Config: {'✅ found' if os.path.exists(cfg) else '❌ missing'}")

### Step 6: Verify Config and Checkpoint Paths

In [ ]:
CONFIG = '/content/mmdetection3d/configs/pointpillars/pointpillars_hv_fpn_sbn-all_8xb4-2x_nus-3d.py'
CHECKPOINT = '/content/pointpillars_weights/pointpillars_nuscenes.pth'

# Verify both exist
import os
print(f"Config    : {'✅' if os.path.exists(CONFIG) else '❌'} {CONFIG}")
print(f"Checkpoint: {'✅' if os.path.exists(CHECKPOINT) else '❌'} {CHECKPOINT}")

## Symlink NuScenes to MMDet3D

In [ ]:
import os

# Create the directory structure it expects
os.makedirs('/content/mmdetection3d/data', exist_ok=True)

# Symlink your nuScenes to where MMDet3D expects it
if not os.path.exists('/content/mmdetection3d/data/nuscenes'):
    os.symlink('/data/sets/nuscenes', '/content/mmdetection3d/data/nuscenes')
    print("✅ Symlink created")
else:
    print("✅ Already exists")

# Verify
print(os.path.exists('/content/mmdetection3d/data/nuscenes/v1.0-mini'))

## Dataset Preparation for PointPillars 3D using Nuscenes mini LiDAR dataset

In [ ]:
%cd /content/mmdetection3d
!PYTHONPATH=/content/mmdetection3d:$PYTHONPATH python tools/create_data.py nuscenes \
    --root-path data/nuscenes \
    --out-dir data/nuscenes \
    --extra-tag nuscenes \
    --version v1.0-mini

### Load PointPillars Model

In [ ]:
import sys
sys.path.insert(0, '/content/mmdetection3d')

from mmdet3d.apis import init_model, inference_detector
from mmdet3d.utils import register_all_modules

register_all_modules()

# Load model
model = init_model(CONFIG, CHECKPOINT, device='cuda:0')
print("✅ Model loaded")

### Select a LiDAR Sample

In [ ]:
from nuscenes.nuscenes import NuScenes
import os

NUSCENES_ROOT = '/data/sets/nuscenes'
nusc = NuScenes(version='v1.0-mini', dataroot=NUSCENES_ROOT, verbose=False)

# Get Sample 1 LiDAR file
sample = nusc.sample[1]
lidar_token = sample['data']['LIDAR_TOP']
lidar_data  = nusc.get('sample_data', lidar_token)
lidar_path  = os.path.join(NUSCENES_ROOT, lidar_data['filename'])

print(f"LiDAR file: {lidar_path}")
print(f"Exists    : {os.path.exists(lidar_path)}")

## Run PointPillars on the LiDAR point cloud

In [ ]:
result, data = inference_detector(model, lidar_path)
print("✅ Inference done")

# Check output
boxes  = result.pred_instances_3d.bboxes_3d
scores = result.pred_instances_3d.scores_3d
labels = result.pred_instances_3d.labels_3d

print(f"Detected {len(boxes)} objects")
print(f"Scores  : {scores[:5]}")
print(f"Labels  : {labels[:5]}")

## NuScenes Mini PointPillars 3D Detection BEV Visualization

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# nuScenes class names for PointPillars
CLASS_NAMES = [
    'car', 'truck', 'construction_vehicle', 'bus', 'trailer',
    'barrier', 'motorcycle', 'bicycle', 'pedestrian', 'traffic_cone'
]

CLASS_COLORS = {
    'car': 'cyan', 'truck': 'orange', 'bus': 'yellow',
    'pedestrian': 'red', 'motorcycle': 'lime', 'bicycle': 'lime',
    'traffic_cone': 'white', 'barrier': 'magenta',
    'construction_vehicle': 'orange', 'trailer': 'orange'
}

# Filter by confidence score
SCORE_THRESH = 0.3
mask    = scores.cpu().numpy() > SCORE_THRESH
boxes_np  = boxes.tensor.cpu().numpy()[mask]
scores_np = scores.cpu().numpy()[mask]
labels_np = labels.cpu().numpy()[mask]

print(f"Detections above {SCORE_THRESH} threshold: {mask.sum()}")

# Plot BEV
fig, ax = plt.subplots(1, 1, figsize=(12, 12))
ax.set_facecolor('black')
ax.set_xlim(-50, 50)
ax.set_ylim(-50, 50)
ax.set_xlabel('X (meters)', color='white')
ax.set_ylabel('Y (meters)', color='white')
ax.set_title('PointPillars 3D Detection — Bird\'s Eye View', color='white')
ax.tick_params(colors='white')
ax.grid(True, color='gray', alpha=0.3)

# Draw ego vehicle at center
ego = patches.Rectangle((-1, -2), 2, 4, color='white', zorder=5)
ax.add_patch(ego)
ax.text(0, 0, 'EGO', color='black', ha='center', va='center',
        fontsize=8, fontweight='bold', zorder=6)

# Draw each detected box
for i in range(len(boxes_np)):
    x, y, z, w, l, h, yaw = boxes_np[i][:7]
    cls_name  = CLASS_NAMES[labels_np[i]] if labels_np[i] < len(CLASS_NAMES) else 'unknown'
    color     = CLASS_COLORS.get(cls_name, 'white')
    score     = scores_np[i]

    # Rotate box corners based on heading angle
    cos_yaw, sin_yaw = np.cos(yaw), np.sin(yaw)
    corners = np.array([
        [-l/2, -w/2], [ l/2, -w/2],
        [ l/2,  w/2], [-l/2,  w/2]
    ])
    rot = np.array([[cos_yaw, -sin_yaw], [sin_yaw, cos_yaw]])
    corners = (rot @ corners.T).T + np.array([x, y])

    poly = patches.Polygon(corners, closed=True,
                           edgecolor=color, facecolor=color,
                           alpha=0.3, linewidth=1.5)
    ax.add_patch(poly)

    # Label with class + score
    ax.text(x, y, f'{cls_name[:3]}\n{score:.2f}',
            color=color, fontsize=6, ha='center', va='center')

# Legend
legend_items = [patches.Patch(color=c, label=n)
                for n, c in CLASS_COLORS.items()]
ax.legend(handles=legend_items, loc='upper right',
          facecolor='black', labelcolor='white', fontsize=8)

fig.patch.set_facecolor('black')
plt.tight_layout()
plt.savefig('/content/bev_detection.jpg', dpi=150,
            bbox_inches='tight', facecolor='black')
plt.show()
print(f"✅ BEV saved → /content/bev_detection.jpg")
print(f"\nDetection summary:")
for cls_id, cls_name in enumerate(CLASS_NAMES):
    count = (labels_np == cls_id).sum()
    if count > 0:
        print(f"  {cls_name:<25} {count}")

# NuScenes Mini Point Cloud vs PointPillars 3D Detection BEV Visualization

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Load raw point cloud
points = np.fromfile(lidar_path, dtype=np.float32).reshape(-1, 5)
# nuScenes LiDAR format: x, y, z, intensity, ring_index
x, y, z = points[:, 0], points[:, 1], points[:, 2]

fig, axes = plt.subplots(1, 2, figsize=(20, 10))
fig.patch.set_facecolor('black')

# Left: raw point cloud BEV
ax1 = axes[0]
ax1.set_facecolor('black')
ax1.scatter(x, y, s=0.3, c=z, cmap='viridis', alpha=0.6)
ax1.set_xlim(-50, 50)
ax1.set_ylim(-50, 50)
ax1.set_title('Raw LiDAR Point Cloud (BEV)', color='white')
ax1.set_xlabel('X (meters)', color='white')
ax1.set_ylabel('Y (meters)', color='white')
ax1.tick_params(colors='white')
ax1.plot(0, 0, 'ws', markersize=8)  # ego vehicle
ax1.text(0, -3, 'EGO', color='white', ha='center', fontsize=8)

# Right: PointPillars detections BEV (replot)
ax2 = axes[1]
ax2.set_facecolor('black')
ax2.set_xlim(-50, 50)
ax2.set_ylim(-50, 50)
ax2.set_title('PointPillars 3D Detections (BEV)', color='white')
ax2.set_xlabel('X (meters)', color='white')
ax2.tick_params(colors='white')
ax2.grid(True, color='gray', alpha=0.3)

ego = patches.Rectangle((-1, -2), 2, 4, color='white', zorder=5)
ax2.add_patch(ego)
ax2.text(0, 0, 'EGO', color='black', ha='center', va='center',
         fontsize=8, fontweight='bold', zorder=6)

for i in range(len(boxes_np)):
    x_b, y_b, z_b, w, l, h, yaw = boxes_np[i][:7]
    cls_name = CLASS_NAMES[labels_np[i]] if labels_np[i] < len(CLASS_NAMES) else 'unknown'
    color    = CLASS_COLORS.get(cls_name, 'white')
    cos_yaw, sin_yaw = np.cos(yaw), np.sin(yaw)
    corners  = np.array([[-l/2,-w/2],[l/2,-w/2],[l/2,w/2],[-l/2,w/2]])
    rot      = np.array([[cos_yaw,-sin_yaw],[sin_yaw,cos_yaw]])
    corners  = (rot @ corners.T).T + np.array([x_b, y_b])
    poly     = patches.Polygon(corners, closed=True,
                               edgecolor=color, facecolor=color,
                               alpha=0.4, linewidth=1.5)
    ax2.add_patch(poly)
    ax2.text(x_b, y_b, f'{cls_name[:3]}\n{scores_np[i]:.2f}',
             color=color, fontsize=6, ha='center', va='center')

plt.tight_layout()
plt.savefig('/content/pointcloud_vs_detections.jpg', dpi=150,
            bbox_inches='tight', facecolor='black')
plt.show()
print("✅ Saved → /content/pointcloud_vs_detections.jpg")

# NuScenes Mini PointPillars 3D Detection with Raw Point Cloud BEV Visualization

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Load raw point cloud for background
points  = np.fromfile(lidar_path, dtype=np.float32).reshape(-1, 5)
px, py, pz = points[:, 0], points[:, 1], points[:, 2]

fig, ax = plt.subplots(1, 1, figsize=(14, 14))
fig.patch.set_facecolor('#0a0a0a')
ax.set_facecolor('#0a0a0a')

# ── Background: LiDAR point cloud ────────────────────────────────
# Filter to ground + slightly above (z between -3 and 2)
mask_z = (pz > -3) & (pz < 2)
# Color by height: ground=dark, objects=bright
ax.scatter(px[mask_z], py[mask_z], s=0.2,
           c=pz[mask_z], cmap='plasma',
           alpha=0.4, vmin=-2, vmax=1, zorder=1)

# ── Range rings (like radar display) ─────────────────────────────
for r in [10, 20, 30, 40, 50]:
    circle = plt.Circle((0, 0), r, color='#333333',
                         fill=False, linewidth=0.5, zorder=2)
    ax.add_patch(circle)
    ax.text(r + 0.5, 0.5, f'{r}m', color='#555555',
            fontsize=7, zorder=2)

# ── Forward direction arrow ───────────────────────────────────────
ax.annotate('', xy=(0, 8), xytext=(0, 3),
            arrowprops=dict(arrowstyle='->', color='#888888', lw=1.5),
            zorder=3)

# ── Detection boxes ───────────────────────────────────────────────
for i in range(len(boxes_np)):
    x, y, z, w, l, h, yaw = boxes_np[i][:7]
    cls_name = CLASS_NAMES[labels_np[i]] if labels_np[i] < len(CLASS_NAMES) else 'unknown'
    color    = CLASS_COLORS.get(cls_name, 'white')
    score    = scores_np[i]

    cos_yaw, sin_yaw = np.cos(yaw), np.sin(yaw)
    corners = np.array([[-l/2,-w/2],[l/2,-w/2],[l/2,w/2],[-l/2,w/2]])
    rot     = np.array([[cos_yaw,-sin_yaw],[sin_yaw,cos_yaw]])
    corners = (rot @ corners.T).T + np.array([x, y])

    # Filled box with stronger edge
    poly = patches.Polygon(corners, closed=True,
                           edgecolor=color, facecolor=color,
                           alpha=0.25, linewidth=2, zorder=4)
    ax.add_patch(poly)

    # Heading line — shows which way object is facing
    front_center = np.mean(corners[1:3], axis=0)
    ax.plot([x, front_center[0]], [y, front_center[1]],
            color=color, linewidth=1.5, alpha=0.9, zorder=5)

    # Label
    ax.text(x, y, f'{cls_name[:3]}\n{score:.2f}',
            color=color, fontsize=6.5, ha='center', va='center',
            fontweight='bold', zorder=6)

# ── Ego vehicle ───────────────────────────────────────────────────
ego = patches.FancyBboxPatch((-1, -2), 2, 4,
                              boxstyle="round,pad=0.1",
                              edgecolor='white', facecolor='#333333',
                              linewidth=2, zorder=7)
ax.add_patch(ego)
ax.text(0, 0, 'EGO', color='white', ha='center', va='center',
        fontsize=8, fontweight='bold', zorder=8)

# ── Styling ───────────────────────────────────────────────────────
ax.set_xlim(-50, 50)
ax.set_ylim(-50, 50)
ax.set_xlabel('X (meters)', color='#aaaaaa', fontsize=11)
ax.set_ylabel('Y (meters)', color='#aaaaaa', fontsize=11)
ax.set_title('PointPillars 3D Detection — Bird\'s Eye View\n'
             'nuScenes Mini | Pre-trained weights',
             color='white', fontsize=13, pad=15)
ax.tick_params(colors='#888888')
for spine in ax.spines.values():
    spine.set_edgecolor('#333333')

# Legend
legend_items = [patches.Patch(color=c, label=n)
                for n, c in CLASS_COLORS.items()
                if any(CLASS_NAMES[l] == n for l in labels_np)]
ax.legend(handles=legend_items, loc='upper left',
          facecolor='#1a1a1a', labelcolor='white',
          fontsize=9, framealpha=0.8)

plt.tight_layout()
plt.savefig('/content/bev_detection_v2.jpg', dpi=150,
            bbox_inches='tight', facecolor='#0a0a0a')
plt.show()
print("✅ Saved → /content/bev_detection_v2.jpg")

### Save Inference Results to JSON

In [ ]:
import json
import numpy as np

# Convert numpy types to Python native types for JSON serialization
def convert_to_serializable(obj):
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj

results_path = '/content/pointpillars_results.json'
with open(results_path, 'w') as f:
    json.dump(results_dict, f, default=convert_to_serializable)
print(f"✅ Results saved → {results_path}")

### Official NuScenes mini_val Evaluation
Runs inference on all official `mini_val` samples, transforms predictions from LiDAR frame to global frame, then evaluates with the nuScenes detection metric (mAP + NDS).

In [ ]:
from nuscenes.utils.splits import create_splits_scenes
from nuscenes.eval.detection.config import config_factory
from nuscenes.eval.detection.evaluate import NuScenesEval
import json, math, os
import numpy as np

# Get official mini_val sample tokens
splits = create_splits_scenes()
mini_val_scenes = splits['mini_val']
print(f"Official mini_val scenes: {mini_val_scenes}")

# Collect all sample tokens from official mini_val scenes
val_samples = []
for scene in nusc.scene:
    if scene['name'] in mini_val_scenes:
        token = scene['first_sample_token']
        while token:
            val_samples.append(nusc.get('sample', token))
            token = nusc.get('sample', token)['next']

print(f"Official mini_val samples: {len(val_samples)}")

# Rerun inference on correct samples
print("Running inference...")
all_results = []

CLASS_NAMES = ['car','truck','trailer','bus','construction_vehicle',
               'bicycle','motorcycle','pedestrian','traffic_cone','barrier']

for i, sample in enumerate(val_samples):
    lidar_token = sample['data']['LIDAR_TOP']
    lidar_data  = nusc.get('sample_data', lidar_token)
    lidar_path  = os.path.join('/data/sets/nuscenes', lidar_data['filename'])

    result, _ = inference_detector(model, lidar_path)

    boxes  = result.pred_instances_3d.bboxes_3d.tensor.cpu().numpy()
    scores = result.pred_instances_3d.scores_3d.cpu().numpy()
    labels = result.pred_instances_3d.labels_3d.cpu().numpy()

    ep = nusc.get('ego_pose',
                  nusc.get('sample_data',
                           sample['data']['LIDAR_TOP'])['ego_pose_token'])

    for j in range(len(boxes)):
        x, y, z, w, l, h, yaw = boxes[j][:7]
        score = float(scores[j])
        label = int(labels[j])
        if score < 0.05 or label >= len(CLASS_NAMES):
            continue

        ego_yaw   = math.atan2(
            2*(ep['rotation'][0]*ep['rotation'][3]),
            1-2*ep['rotation'][3]**2)
        cos_e, sin_e = math.cos(ego_yaw), math.sin(ego_yaw)
        gx   = cos_e*x - sin_e*y + ep['translation'][0]
        gy   = sin_e*x + cos_e*y + ep['translation'][1]
        gz   = z + ep['translation'][2]
        gyaw = yaw + ego_yaw

        all_results.append({
            'sample_token':    sample['token'],
            'translation':     [float(gx), float(gy), float(gz)],
            'size':            [float(w), float(l), float(h)],
            'rotation':        [float(math.cos(gyaw/2)), 0.0, 0.0,
                                float(math.sin(gyaw/2))],
            'velocity':        [0.0, 0.0],
            'detection_name':  CLASS_NAMES[label],
            'detection_score': float(score),
            'attribute_name':  ''
        })

    if (i+1) % 20 == 0:
        print(f"  {i+1}/{len(val_samples)} done")

print(f"✅ Total predictions: {len(all_results)}")

# Build results dict — include ALL val sample tokens (even empty ones)
results_dict = {
    'meta': {'use_lidar': True, 'use_camera': False,
             'use_radar': False, 'use_map': False, 'use_external': False},
    'results': {}
}
for r in all_results:
    token = r['sample_token']
    if token not in results_dict['results']:
        results_dict['results'][token] = []
    results_dict['results'][token].append(r)

# Fill missing samples with empty predictions
for sample in val_samples:
    if sample['token'] not in results_dict['results']:
        results_dict['results'][sample['token']] = []

results_path = '/content/pointpillars_results.json'
with open(results_path, 'w') as f:
    json.dump(results_dict, f)
print(f"✅ Saved {len(results_dict['results'])} sample results")

# Run evaluation
nusc_eval = NuScenesEval(
    nusc,
    config=config_factory('detection_cvpr_2019'),
    result_path=results_path,
    eval_set='mini_val',
    output_dir='/content/pointpillars_eval',
    verbose=True
)
metrics, _ = nusc_eval.evaluate()

print(f"\n{'='*50}")
print(f"PointPillars — nuScenes Mini Val Results")
print(f"{'='*50}")
print(f"mAP : {metrics.mean_ap:.4f}")
print(f"NDS : {metrics.nd_score:.4f}")
print(f"\nPer-class AP:")
for cls, ap in metrics.mean_dist_aps.items():
    print(f"  {cls:<25} {ap:.4f}")

## Model Export — ONNX and TFLite
Export all three trained models (YOLO26n-det, YOLO26n-seg, YOLOv8n-det) to ONNX, TFLite FP16, and TFLite INT8 for edge deployment.

In [ ]:
!pip install ultralytics --quiet
!pip install torch-pruning --quiet  # for structured pruning

### Mount Drive and Verify Model Weights

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

RUNS_DIR = '/content/drive/MyDrive/yolo_runs'  # adjust if different

models = {
    'yolo26n_det': f'{RUNS_DIR}/yolo26n_nuscenes/weights/best.pt',
    'yolo26n_seg': f'{RUNS_DIR}/yolo26n_seg_nuscenes/weights/best.pt',
    'yolov8n_det': f'{RUNS_DIR}/yolov8n_nuscenes/weights/best.pt',
}

for name, path in models.items():
    status = "✅" if os.path.exists(path) else "❌ NOT FOUND"
    print(f"{status}  {name}: {path}")

### Verify Dataset YAML and nuScenes Root

In [ ]:
NUSCENES_ROOT = '/data/sets/nuscenes'
YAML_PATH     = '/content/drive/MyDrive/yolo_runs/nuscenes_yolo/nuscenes.yaml'

print(f"nuScenes : {'✅' if os.path.exists(NUSCENES_ROOT) else '❌'}")
print(f"YAML     : {'✅' if os.path.exists(YAML_PATH) else '❌'}")

### Load Model and Inspect Parameter Count

In [ ]:
from ultralytics import YOLO

model = YOLO(models['yolo26n_det'])
print(f"✅ YOLO26n-det loaded")
print(f"   Parameters: {sum(p.numel() for p in model.model.parameters()):,}")

### Export All Models to ONNX and TFLite (FP16 and INT8)

In [ ]:
# Export Model to ONNX and TFLite
from ultralytics import YOLO
import os

RUNS_DIR  = '/content/drive/MyDrive/yolo_runs'
EXPORT_DIR = '/content/drive/MyDrive/yolo_runs/exports'
os.makedirs(EXPORT_DIR, exist_ok=True)

models = {
    'yolo26n_det': f'{RUNS_DIR}/yolo26n_nuscenes/weights/best.pt',
    'yolo26n_seg': f'{RUNS_DIR}/yolo26n_seg_nuscenes/weights/best.pt',
    'yolov8n_det': f'{RUNS_DIR}/yolov8n_nuscenes/weights/best.pt',
}

# Export each model to ONNX and TFLite
for name, path in models.items():
    print(f"\n{'='*40}")
    print(f"Exporting {name}...")
    print(f"{'='*40}")
    model = YOLO(path)

    # ONNX export
    print(f"→ ONNX...")
    onnx_path = model.export(
        format='onnx',
        imgsz=640,
        opset=17,
        simplify=True,
        dynamic=False,
    )
    print(f"✅ ONNX: {onnx_path}")

    # TFLite FP16 export
    print(f"→ TFLite FP16...")
    tflite_path = model.export(
        format='tflite',
        imgsz=640,
        half=True,     # FP16
    )
    print(f"✅ TFLite FP16: {tflite_path}")

    # TFLite INT8 export
    print(f"→ TFLite INT8...")
    tflite_int8_path = model.export(
        format='tflite',
        imgsz=640,
        int8=True,
        data='/content/drive/MyDrive/yolo_runs/nuscenes_yolo/nuscenes.yaml',
    )
    print(f"✅ TFLite INT8: {tflite_int8_path}")

print(f"\n✅ All exports done")

### Fix: Recreate Image Symlinks
Run this cell only if validation fails with a missing-image error. Recreates the `/content` symlinks using the same seed=42 split.

In [ ]:
# If symlink error, Recreate image symlink
import os
import random
import numpy as np
from pathlib import Path
from nuscenes.nuscenes import NuScenes
from pyquaternion import Quaternion

NUSCENES_ROOT = '/data/sets/nuscenes'
OUTPUT_DIR    = '/content/nuscenes_yolo'

# Recreate full directory structure
for split in ['train', 'val']:
    Path(f'{OUTPUT_DIR}/images/{split}').mkdir(parents=True, exist_ok=True)
    Path(f'{OUTPUT_DIR}/labels/{split}').mkdir(parents=True, exist_ok=True)

nusc = NuScenes(version='v1.0-mini', dataroot=NUSCENES_ROOT, verbose=False)

# Recreate same 80/20 split with seed=42
all_samples = []
for scene in nusc.scene:
    token = scene['first_sample_token']
    while token:
        all_samples.append(nusc.get('sample', token))
        token = nusc.get('sample', token)['next']

random.seed(42)
random.shuffle(all_samples)
split_idx = int(len(all_samples) * 0.8)
splits = {
    'train': all_samples[:split_idx],
    'val':   all_samples[split_idx:]
}

# Recreate image symlinks only (labels already exist on Drive)
for split, samples in splits.items():
    for sample in samples:
        cam_data = nusc.get('sample_data', sample['data']['CAM_FRONT'])
        img_path = os.path.join(NUSCENES_ROOT, cam_data['filename'])
        img_name = os.path.basename(img_path)
        dst      = f'{OUTPUT_DIR}/images/{split}/{img_name}'
        if not os.path.exists(dst):
            os.symlink(img_path, dst)

print(f"✅ Image symlinks recreated")
print(f"   train: {len(list(Path(f'{OUTPUT_DIR}/images/train').glob('*.jpg')))} images")
print(f"   val  : {len(list(Path(f'{OUTPUT_DIR}/images/val').glob('*.jpg')))} images")

### Fix: Copy Labels from Drive
Run this cell only if label files are missing from `/content`. Copies them back from Drive without re-running the full conversion.

In [ ]:
# Copy labels to /content for a faster validation
import shutil

# Copy labels from Drive back to /content
for split in ['train', 'val']:
    src = f'/content/drive/MyDrive/yolo_runs/nuscenes_yolo/labels/{split}'
    dst = f'{OUTPUT_DIR}/labels/{split}'

    if os.path.exists(src):
        # Copy all label files
        for f in Path(src).glob('*.txt'):
            shutil.copy(f, f'{dst}/{f.name}')
        count = len(list(Path(dst).glob('*.txt')))
        print(f"✅ {split}: {count} label files copied")
    else:
        print(f"❌ {split}: source not found at {src}")

## Model Validation Across All Export Formats
Runs `model.val()` on every model×format combination and prints a comparison table of mAP50 scores to quantify the accuracy cost of quantization.

In [ ]:
from ultralytics import YOLO

RUNS_DIR = '/content/drive/MyDrive/yolo_runs'
YAML_DET = '/content/nuscenes_yolo/nuscenes.yaml'
YAML_SEG = '/content/drive/MyDrive/yolo_runs/nuscenes_yolo_seg/nuscenes_seg.yaml'

results = {}

models_to_validate = {
    # YOLO26n-det
    'yolo26n_det_FP32':  (f'{RUNS_DIR}/yolo26n_nuscenes/weights/best.pt',               YAML_DET, 'detect'),
    'yolo26n_det_ONNX':  (f'{RUNS_DIR}/yolo26n_nuscenes/weights/best.onnx',              YAML_DET, 'detect'),
    'yolo26n_det_FP16':  (f'{RUNS_DIR}/yolo26n_nuscenes/weights/best_saved_model/best_float16.tflite', YAML_DET, 'detect'),
    'yolo26n_det_INT8':  (f'{RUNS_DIR}/yolo26n_nuscenes/weights/best_saved_model/best_int8.tflite',    YAML_DET, 'detect'),
    # YOLO26n-seg
    'yolo26n_seg_FP32':  (f'{RUNS_DIR}/yolo26n_seg_nuscenes/weights/best.pt',            YAML_SEG, 'segment'),
    'yolo26n_seg_ONNX':  (f'{RUNS_DIR}/yolo26n_seg_nuscenes/weights/best.onnx',          YAML_SEG, 'segment'),
    'yolo26n_seg_FP16':  (f'{RUNS_DIR}/yolo26n_seg_nuscenes/weights/best_saved_model/best_float16.tflite', YAML_SEG, 'segment'),
    'yolo26n_seg_INT8':  (f'{RUNS_DIR}/yolo26n_seg_nuscenes/weights/best_saved_model/best_int8.tflite',    YAML_SEG, 'segment'),
    # YOLOv8n-det
    'yolov8n_det_FP32':  (f'{RUNS_DIR}/yolov8n_nuscenes/weights/best.pt',                YAML_DET, 'detect'),
    'yolov8n_det_ONNX':  (f'{RUNS_DIR}/yolov8n_nuscenes/weights/best.onnx',              YAML_DET, 'detect'),
    'yolov8n_det_FP16':  (f'{RUNS_DIR}/yolov8n_nuscenes/weights/best_saved_model/best_float16.tflite', YAML_DET, 'detect'),
    'yolov8n_det_INT8':  (f'{RUNS_DIR}/yolov8n_nuscenes/weights/best_saved_model/best_int8.tflite',    YAML_DET, 'detect'),
}

for name, (path, yaml, task) in models_to_validate.items():
    print(f"\nValidating {name}...")
    try:
        model   = YOLO(path)
        metrics = model.val(data=yaml, imgsz=640, verbose=False)
        map50   = metrics.box.map50
        results[name] = map50
        print(f"  ✅ mAP50: {map50:.4f}")
    except Exception as e:
        results[name] = None
        print(f"  ❌ Failed: {e}")

# Print final table
print(f"\n{'='*55}")
print(f"{'Model':<25} {'Format':<10} {'mAP50':>8}")
print(f"{'='*55}")
for name, map50 in results.items():
    parts  = name.rsplit('_', 1)
    model  = parts[0]
    fmt    = parts[1]
    val    = f"{map50:.4f}" if map50 is not None else "FAILED"
    print(f"{model:<25} {fmt:<10} {val:>8}")

## Quantization-Aware Training (QAT)
Fine-tunes the YOLO26n-det model with fake quantization nodes active (`int8=True`), so weights adapt to INT8 precision before export — typically recovers 1-3% mAP vs. post-training quantization.

In [ ]:
# Train YOLO26 with INT8 QAT
from ultralytics import YOLO

RUNS_DIR = '/content/drive/MyDrive/yolo_runs'
YAML_DET = '/content/nuscenes_yolo/nuscenes.yaml'

# QAT on YOLO26n-det
model = YOLO(f'{RUNS_DIR}/yolo26n_nuscenes/weights/best.pt')

model.train(
    data=YAML_DET,
    epochs=20,           # short fine-tuning, not full training
    imgsz=640,
    batch=16,
    name='yolo26n_nuscenes_qat',
    project=RUNS_DIR,
    device=0,
    workers=2,
    patience=10,
    save=True,
    plots=True,
    int8=True,           # ← enables QAT
)

## Structured Pruning
Removes entire channels from Conv2d layers based on L1-norm importance, then adjusts dependent layers automatically. Target: 30% parameter reduction with minimal accuracy loss.

In [ ]:
!pip install torch-pruning --quiet      # Install torch-pruning for structured pruning

### Inspect Prunable Layers
Identifies which Conv2d layers torch-pruning can handle and which custom YOLO blocks (C3k2, SPPF, Detect head) will be excluded.

In [ ]:
# Check what torch-pruning can actually prune
prunable = []
ignored  = []

for name, module in model.named_modules():
    if isinstance(module, torch.nn.Conv2d):
        prunable.append(f"{name}: {module.in_channels}→{module.out_channels}")
    elif hasattr(module, '__class__'):
        cname = module.__class__.__name__
        if cname in ['C3k2', 'SPPF', 'C2PSA', 'Detect']:
            ignored.append(f"{name}: {cname}")

print(f"Conv2d layers found: {len(prunable)}")
for p in prunable[:5]:
    print(f"  {p}")

print(f"\nCustom blocks (not recognized): {len(ignored)}")
for i in ignored[:5]:
    print(f"  {i}")

### Apply Structured Pruning
Builds the dependency graph, prunes 30% of channels from all eligible Conv2d layers (skipping the detection head), then verifies the forward pass still works.

In [ ]:
# Try structured pruning using torch_pruning
import torch
import torch_pruning as tp
from ultralytics import YOLO

RUNS_DIR = '/content/drive/MyDrive/yolo_runs'
YAML_DET = '/content/nuscenes_yolo/nuscenes.yaml'

model_path = f'{RUNS_DIR}/yolo26n_nuscenes/weights/best.pt'
yolo  = YOLO(model_path)
model = yolo.model.eval()

params_before = sum(p.numel() for p in model.parameters())
print(f"Parameters before: {params_before:,}")

# Build dependency graph — this is what makes it truly structured
# It tracks which layers depend on each other so when you remove
# channels from layer N, it automatically adjusts layer N+1
example_input = torch.randn(1, 3, 640, 640).to(next(model.parameters()).device)

importance = tp.importance.MagnitudeImportance(p=1)  # L1 norm

pruner = tp.pruner.MagnitudePruner(
    model,
    example_inputs=example_input,
    importance=importance,
    pruning_ratio=0.3,           # 30% channel reduction
    ignored_layers=[model.model[-1]],  # don't prune the detection head
)

pruner.step()

params_after = sum(p.numel() for p in model.parameters())
print(f"Parameters after : {params_after:,}")
print(f"Reduction        : {(1 - params_after/params_before)*100:.1f}%")

# Verify forward pass still works
with torch.no_grad():
    out = model(example_input)
print(f"✅ Forward pass OK — output shape: {out[0].shape}")

## Camera-LiDAR Late Fusion
Late fusion pipeline that combines 2D camera detections (YOLO26n on CAM_FRONT) with 3D LiDAR detections (PointPillars) in the ego-vehicle frame.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
RUNS_DIR     = '/content/drive/MyDrive/yolo_runs'
NUSCENES_ROOT = '/data/sets/nuscenes'

# Check weights
weights = f'{RUNS_DIR}/yolo26n_nuscenes/weights/best.pt'
print(f"YOLO26n weights : {'✅' if os.path.exists(weights) else '❌'}")
print(f"nuScenes root   : {'✅' if os.path.exists(NUSCENES_ROOT) else '❌'}")

### Load nuScenes and YOLO Model

In [ ]:
from nuscenes.nuscenes import NuScenes
from ultralytics import YOLO

nusc  = NuScenes(version='v1.0-mini', dataroot=NUSCENES_ROOT, verbose=False)
model = YOLO(weights)

print(f"✅ nuScenes loaded — {len(nusc.sample)} samples")
print(f"✅ YOLO26n loaded")

### Inspect Camera Calibration (K Matrix)
Verify the intrinsics from nuScenes metadata — used to back-project 2D detections onto the ground plane.

In [ ]:
import numpy as np
import os
from pyquaternion import Quaternion
from nuscenes.nuscenes import NuScenes

NUSCENES_ROOT = '/data/sets/nuscenes'
nusc = NuScenes(version='v1.0-mini', dataroot=NUSCENES_ROOT, verbose=False)

# Get sample 1 and inspect camera calibration
sample     = nusc.sample[1]
cam_token  = sample['data']['CAM_FRONT']
cam_data   = nusc.get('sample_data', cam_token)
cs         = nusc.get('calibrated_sensor',
                      cam_data['calibrated_sensor_token'])

print("Camera intrinsics K:")
K = np.array(cs['camera_intrinsic'])
print(K)
print(f"\nfx={K[0,0]:.1f}  fy={K[1,1]:.1f}")
print(f"cx={K[0,2]:.1f}  cy={K[1,2]:.1f}")

print(f"\nCamera→Ego translation: {cs['translation']}")
print(f"Camera→Ego rotation   : {cs['rotation']}")

# Image size
print(f"\nImage size: {cam_data['width']}×{cam_data['height']}")

### Run YOLO26n Camera Detection on CAM_FRONT
Detects objects in the front camera image and projects each bounding box bottom-center to a BEV ego-frame position via ground-plane intersection.

In [ ]:
import numpy as np
from pyquaternion import Quaternion
from ultralytics import YOLO
import os

# ── Camera calibration (nuScenes CAM_FRONT) ───────────────────────────────────
K = np.array([
    [1266.4,    0.0, 816.27],
    [   0.0, 1266.4, 491.51],
    [   0.0,    0.0,   1.0 ]
])

def bbox_to_bev(bbox_xyxy, K, cam_translation, cam_rotation,
                camera_height=1.51):
    """
    Project a 2D bounding box bottom-center to BEV ego coordinates.

    Approach: ground plane intersection
        1. Take bottom-center pixel (x, y_bottom) — feet of object
        2. Unproject to camera ray: d = K_inv @ [u, v, 1]
        3. Scale ray to hit ground plane z=0 in camera frame
           Camera is at height h above ground, so scale = h / d[2]
        4. Rotate + translate from camera frame → ego frame

    Args:
        bbox_xyxy      : [x1, y1, x2, y2] in pixels
        K              : 3×3 camera intrinsic matrix
        cam_translation: [x, y, z] camera position in ego frame (meters)
        cam_rotation   : [w, x, y, z] quaternion camera→ego rotation
        camera_height  : camera height above ground (meters)
                         nuScenes CAM_FRONT z = 1.51m

    Returns:
        np.ndarray [x, y] position in ego BEV frame (meters)
        None if projection is behind camera or invalid
    """
    x1, y1, x2, y2 = bbox_xyxy

    # Bottom-center pixel — ground contact point of object
    u = (x1 + x2) / 2.0
    v = y2   # bottom of bounding box

    # Unproject: image → normalized camera ray
    K_inv = np.linalg.inv(K)
    ray_cam = K_inv @ np.array([u, v, 1.0])

    # ray_cam = [dx, dy, dz] in camera frame
    # Camera frame: x=right, y=down, z=forward
    # Ground plane in camera frame: y = camera_height (since y points down)
    # Scale factor: t = camera_height / ray_cam[1]
    if ray_cam[1] <= 0:
        # Ray points upward — no ground intersection
        return None

    t = camera_height / ray_cam[1]

    if t <= 0:
        return None

    # 3D point in camera frame
    point_cam = ray_cam * t  # [x, y, z] in camera frame

    # Transform camera frame → ego frame
    # Camera frame: x=right, y=down, z=forward
    # Ego frame   : x=forward, y=left, z=up
    R_cam2ego = Quaternion(cam_rotation).rotation_matrix
    t_cam2ego = np.array(cam_translation)

    point_ego = R_cam2ego @ point_cam + t_cam2ego

    # Return x, y in ego BEV (ignore z — should be ~0 for ground objects)
    return point_ego[:2]


def run_camera_to_bev(nusc, model, sample_idx=1,
                      score_thresh=0.3, camera_height=1.51):
    """
    Run YOLO26n on a nuScenes front camera image and project
    all detections to BEV ego coordinates.

    Args:
        nusc         : NuScenes instance
        model        : loaded YOLO model
        sample_idx   : which sample to process
        score_thresh : minimum confidence threshold
        camera_height: camera height above ground (meters)

    Returns:
        detections_bev: list of dicts with keys:
            class_name, score, bbox_px, bev_xy, distance
        image_path: path to the camera image
    """
    sample   = nusc.sample[sample_idx]
    cam_token = sample['data']['CAM_FRONT']
    cam_data  = nusc.get('sample_data', cam_token)
    cs        = nusc.get('calibrated_sensor',
                         cam_data['calibrated_sensor_token'])

    image_path     = os.path.join('/data/sets/nuscenes', cam_data['filename'])
    cam_translation = cs['translation']
    cam_rotation    = cs['rotation']

    # Run YOLO26n inference
    results = model(image_path, verbose=False)[0]

    CLASS_NAMES = ['car', 'pedestrian', 'bicycle', 'motorcycle',
                   'bus', 'truck', 'traffic_cone', 'barrier']

    detections_bev = []
    for box in results.boxes:
        score = float(box.conf[0])
        if score < score_thresh:
            continue

        label    = int(box.cls[0])
        cls_name = CLASS_NAMES[label] if label < len(CLASS_NAMES) else 'unknown'
        bbox     = box.xyxy[0].cpu().numpy()

        bev_xy = bbox_to_bev(bbox, K,
                              cam_translation, cam_rotation,
                              camera_height)
        if bev_xy is None:
            continue

        distance = float(np.sqrt(bev_xy[0]**2 + bev_xy[1]**2))

        detections_bev.append({
            'class_name': cls_name,
            'score':      score,
            'bbox_px':    bbox,
            'bev_xy':     bev_xy,
            'distance':   distance,
        })

    return detections_bev, image_path


# ── Test on sample 1 ──────────────────────────────────────────────────────
RUNS_DIR = '/content/drive/MyDrive/yolo_runs'
model    = YOLO(f'{RUNS_DIR}/yolo26n_nuscenes/weights/best.pt')

detections, img_path = run_camera_to_bev(nusc, model, sample_idx=1)

print(f"Image: {img_path}")
print(f"Detections projected to BEV: {len(detections)}")
print(f"\n{'Class':<15} {'Score':>6} {'BEV x':>8} {'BEV y':>8} {'Dist':>8}")
print("-"*50)
for d in sorted(detections, key=lambda x: x['distance']):
    print(f"{d['class_name']:<15} {d['score']:>6.3f} "
          f"{d['bev_xy'][0]:>8.2f} {d['bev_xy'][1]:>8.2f} "
          f"{d['distance']:>7.1f}m")

### Visualize Camera Detections (Image + BEV)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import numpy as np

CLASS_COLORS = {
    'car': 'cyan', 'truck': 'orange', 'bus': 'yellow',
    'pedestrian': 'red', 'motorcycle': 'lime', 'bicycle': 'lime',
    'traffic_cone': 'white', 'barrier': 'magenta',
    'construction_vehicle': 'orange', 'trailer': 'orange',
}

fig, axes = plt.subplots(1, 2, figsize=(20, 9))
fig.patch.set_facecolor('#0a0a0a')

# ── Left: Camera image with YOLO26n detections ────────────────────────────────
ax1 = axes[0]
img = np.array(Image.open(img_path))
ax1.imshow(img)
ax1.set_title('YOLO26n Detections — CAM_FRONT', color='white', fontsize=13)
ax1.axis('off')

for d in detections:
    x1, y1, x2, y2 = d['bbox_px']
    color = CLASS_COLORS.get(d['class_name'], 'white')
    rect  = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                               linewidth=2, edgecolor=color,
                               facecolor='none')
    ax1.add_patch(rect)
    ax1.text(x1, y1-5, f"{d['class_name'][:3]} {d['score']:.2f}",
             color=color, fontsize=7, fontweight='bold')
    # Mark bottom-center (ground contact point)
    ax1.plot((x1+x2)/2, y2, 'o', color=color, markersize=4)

# ── Right: BEV projection ─────────────────────────────────────────────────────
ax2 = axes[1]
ax2.set_facecolor('#0a0a0a')
ax2.set_xlim(-30, 80)
ax2.set_ylim(-40, 40)
ax2.set_xlabel('X — forward (meters)', color='#aaaaaa')
ax2.set_ylabel('Y — left (meters)', color='#aaaaaa')
ax2.set_title('Camera Detections in BEV', color='white', fontsize=13)
ax2.tick_params(colors='#888888')
for spine in ax2.spines.values():
    spine.set_edgecolor('#333333')

# Range rings
for r in [10, 20, 30, 40, 50]:
    circle = plt.Circle((0, 0), r, color='#333333',
                         fill=False, linewidth=0.5)
    ax2.add_patch(circle)
    ax2.text(r+0.5, 0.5, f'{r}m', color='#555555', fontsize=7)

# Grid
ax2.grid(True, color='#1a1a1a', linewidth=0.5)

# Plot detections — filter beyond 80m
MAX_DIST = 80.0
for d in detections:
    if d['distance'] > MAX_DIST:
        continue
    x, y   = d['bev_xy']
    color  = CLASS_COLORS.get(d['class_name'], 'white')
    ax2.scatter(x, y, c=color, s=80, zorder=5)
    ax2.text(x+0.5, y+0.5,
             f"{d['class_name'][:3]}\n{d['distance']:.0f}m",
             color=color, fontsize=7, zorder=6)

# Ego vehicle
ego = patches.FancyBboxPatch((-1, -1.5), 2, 3,
                              boxstyle="round,pad=0.1",
                              edgecolor='white', facecolor='#333333',
                              linewidth=2, zorder=7)
ax2.add_patch(ego)
ax2.text(0, 0, 'EGO', color='white', ha='center', va='center',
         fontsize=8, fontweight='bold', zorder=8)

# Forward arrow
ax2.annotate('', xy=(6, 0), xytext=(3, 0),
             arrowprops=dict(arrowstyle='->', color='#888888', lw=1.5))

# Legend
legend_items = [patches.Patch(color=c, label=n)
                for n, c in CLASS_COLORS.items()
                if any(d['class_name'] == n for d in detections)]
ax2.legend(handles=legend_items, loc='upper right',
           facecolor='#1a1a1a', labelcolor='white', fontsize=8)

plt.tight_layout()
plt.savefig('/content/camera_bev_projection.jpg',
            dpi=150, bbox_inches='tight', facecolor='#0a0a0a')
plt.show()
print("✅ Saved → /content/camera_bev_projection.jpg")

### Run PointPillars LiDAR Detection
Loads the same sample's LiDAR sweep, runs PointPillars inference, and converts predictions to ego-frame BEV coordinates for fusion.

In [ ]:
import sys, os
sys.path.insert(0, '/content/mmdetection3d')

from mmdet3d.apis import init_model, inference_detector
from mmdet3d.utils import register_all_modules
from nuscenes.nuscenes import NuScenes

register_all_modules()

NUSCENES_ROOT = '/data/sets/nuscenes'
CONFIG     = '/content/mmdetection3d/configs/pointpillars/pointpillars_hv_fpn_sbn-all_8xb4-2x_nus-3d.py'
CHECKPOINT = '/content/pointpillars_weights/pointpillars_nuscenes.pth'

model = init_model(CONFIG, CHECKPOINT, device='cuda:0')
nusc  = NuScenes(version='v1.0-mini', dataroot=NUSCENES_ROOT, verbose=False)

# Get sample 1 — same sample we used in fusion
sample     = nusc.sample[1]
lidar_data = nusc.get('sample_data', sample['data']['LIDAR_TOP'])
lidar_path = os.path.join(NUSCENES_ROOT, lidar_data['filename'])

result, _ = inference_detector(model, lidar_path)

boxes  = result.pred_instances_3d.bboxes_3d.tensor.cpu().numpy()
scores = result.pred_instances_3d.scores_3d.cpu().numpy()
labels = result.pred_instances_3d.labels_3d.cpu().numpy()

POINTPILLARS_CLASSES = [
    'car', 'truck', 'trailer', 'bus', 'construction_vehicle',
    'bicycle', 'motorcycle', 'pedestrian', 'traffic_cone', 'barrier'
]

print("# PointPillars detections — sample 1")
print("real_lidar = [")
for i in range(len(boxes)):
    score = float(scores[i])
    label = int(labels[i])
    if score < 0.3 or label >= len(POINTPILLARS_CLASSES):
        continue
    x, y = float(boxes[i][0]), float(boxes[i][1])
    dist = (x**2 + y**2)**0.5
    if dist > 50:
        continue
    cls = POINTPILLARS_CLASSES[label]
    print(f"    {{'class_name': '{cls}', 'score': {score:.3f}, "
          f"'bev_xy': np.array([{x:.2f}, {y:.2f}]), "
          f"'box_3d': None, 'source': 'lidar'}},")
print("]")

### Transform LiDAR Detections to Ego Frame and Define Fusion Function
 applies the LiDAR sensor extrinsic (rotation + translation) to bring PointPillars BEV positions into the same frame as camera BEV positions.

In [ ]:
from pyquaternion import Quaternion
import numpy as np

def lidar_to_ego(bev_xy, nusc, sample):
    """
    Transform 2D BEV position from LiDAR frame → ego vehicle frame.
    MMDet3D outputs boxes in LiDAR sensor frame.
    Camera BEV projection outputs in ego frame.
    Must transform to same frame before fusion matching.
    """
    lidar_data = nusc.get('sample_data', sample['data']['LIDAR_TOP'])
    cs = nusc.get('calibrated_sensor', lidar_data['calibrated_sensor_token'])

    R = Quaternion(cs['rotation']).rotation_matrix
    t = np.array(cs['translation'])

    # Full 3D transform (z=0 for BEV)
    pt_lidar = np.array([bev_xy[0], bev_xy[1], 0.0])
    pt_ego   = R @ pt_lidar + t

    return pt_ego[:2]  # return x, y only

# Transform all PointPillars detections to ego frame
sample = nusc.sample[1]

real_lidar_raw = [
    {'class_name': 'car',        'score': 0.606, 'bev_xy': np.array([3.57,  41.51])},
    {'class_name': 'car',        'score': 0.535, 'bev_xy': np.array([8.73, -28.76])},
    {'class_name': 'car',        'score': 0.468, 'bev_xy': np.array([-1.66, 35.83])},
    {'class_name': 'car',        'score': 0.385, 'bev_xy': np.array([-32.34,-21.41])},
    {'class_name': 'car',        'score': 0.311, 'bev_xy': np.array([5.61,  31.29])},
    {'class_name': 'truck',      'score': 0.355, 'bev_xy': np.array([7.23,  43.24])},
    {'class_name': 'truck',      'score': 0.317, 'bev_xy': np.array([-4.32, 10.84])},
    {'class_name': 'bus',        'score': 0.445, 'bev_xy': np.array([-4.32, 10.86])},
    {'class_name': 'pedestrian', 'score': 0.489, 'bev_xy': np.array([-1.24,-17.91])},
    {'class_name': 'pedestrian', 'score': 0.399, 'bev_xy': np.array([12.24,-14.15])},
    {'class_name': 'pedestrian', 'score': 0.396, 'bev_xy': np.array([-3.44,-18.09])},
    {'class_name': 'pedestrian', 'score': 0.355, 'bev_xy': np.array([-0.70,-19.30])},
    {'class_name': 'barrier',    'score': 0.473, 'bev_xy': np.array([7.62,  17.17])},
    {'class_name': 'barrier',    'score': 0.471, 'bev_xy': np.array([7.75,  18.82])},
    {'class_name': 'barrier',    'score': 0.450, 'bev_xy': np.array([9.81,  37.28])},
    {'class_name': 'barrier',    'score': 0.438, 'bev_xy': np.array([6.18, -13.72])},
    {'class_name': 'barrier',    'score': 0.426, 'bev_xy': np.array([8.22,  28.73])},
    {'class_name': 'barrier',    'score': 0.425, 'bev_xy': np.array([9.78,  40.72])},
    {'class_name': 'barrier',    'score': 0.417, 'bev_xy': np.array([7.82,  20.27])},
    {'class_name': 'barrier',    'score': 0.391, 'bev_xy': np.array([8.16,  24.70])},
    {'class_name': 'barrier',    'score': 0.383, 'bev_xy': np.array([8.21,  26.74])},
    {'class_name': 'barrier',    'score': 0.380, 'bev_xy': np.array([7.09,  10.74])},
    {'class_name': 'barrier',    'score': 0.379, 'bev_xy': np.array([7.19,  12.74])},
    {'class_name': 'barrier',    'score': 0.377, 'bev_xy': np.array([8.32,  10.32])},
    {'class_name': 'barrier',    'score': 0.374, 'bev_xy': np.array([8.86,  16.81])},
    {'class_name': 'barrier',    'score': 0.374, 'bev_xy': np.array([16.64, 27.73])},
    {'class_name': 'barrier',    'score': 0.369, 'bev_xy': np.array([7.86,  22.27])},
    {'class_name': 'barrier',    'score': 0.357, 'bev_xy': np.array([7.03,   7.22])},
    {'class_name': 'barrier',    'score': 0.356, 'bev_xy': np.array([7.35,  14.75])},
    {'class_name': 'barrier',    'score': 0.341, 'bev_xy': np.array([9.30,  30.77])},
    {'class_name': 'barrier',    'score': 0.339, 'bev_xy': np.array([8.79,  34.22])},
    {'class_name': 'barrier',    'score': 0.331, 'bev_xy': np.array([8.21,   6.71])},
    {'class_name': 'barrier',    'score': 0.307, 'bev_xy': np.array([7.01,   9.27])},
]

# Transform to ego frame
real_lidar = []
for d in real_lidar_raw:
    ego_xy = lidar_to_ego(d['bev_xy'], nusc, sample)
    real_lidar.append({
        'class_name': d['class_name'],
        'score':      d['score'],
        'bev_xy':     ego_xy,
        'box_3d':     None,
        'source':     'lidar',
    })

# Print transformed positions to verify
print("PointPillars detections in EGO frame:")
print(f"\n{'Class':<15} {'Score':>6} {'EGO x':>8} {'EGO y':>8} {'Dist':>8}")
print("-"*50)
for d in sorted(real_lidar, key=lambda x: float(np.linalg.norm(x['bev_xy']))):
    dist = float(np.linalg.norm(d['bev_xy']))
    if dist > 60:
        continue
    x, y = d['bev_xy']
    print(f"{d['class_name']:<15} {d['score']:>6.3f} {x:>8.2f} {y:>8.2f} {dist:>7.1f}m")

### Run Initial Late Fusion (3m Match Threshold)
Matches camera and LiDAR detections using nearest-neighbour in BEV ego space. A match is accepted when the distance is below the threshold.

In [ ]:
# Filter camera detections to 60m
camera_dets_filtered = [d for d in detections if d['distance'] <= 60.0]

# Run fusion with REAL LiDAR
fused = fuse_detections(camera_dets_filtered, real_lidar)

n_fused  = sum(1 for d in fused if d['source'] == 'fused')
n_lidar  = sum(1 for d in fused if d['source'] == 'lidar')
n_camera = sum(1 for d in fused if d['source'] == 'camera')

print(f"REAL Fusion results (sample 1):")
print(f"  Camera dets (≤60m) : {len(camera_dets_filtered)}")
print(f"  LiDAR dets         : {len(real_lidar)}")
print(f"  ─────────────────────────────")
print(f"  Fused matches      : {n_fused}")
print(f"  LiDAR-only         : {n_lidar}")
print(f"  Camera-only        : {n_camera}")
print(f"  Total output       : {len(fused)}")

print(f"\n  {'Source':<8} {'Class':<15} {'Score':>6} {'x':>8} {'y':>8} {'Dist':>8}")
print(f"  {'-'*58}")
for d in sorted(fused,
                key=lambda x: float(np.linalg.norm(x['bev_xy']))):
    dist = float(np.linalg.norm(d['bev_xy']))
    if dist > 60:
        continue
    x, y = d['bev_xy']
    print(f"  {d['source']:<8} {d['class_name']:<15} "
          f"{d['score']:>6.3f} {x:>8.2f} {y:>8.2f} {dist:>7.1f}m")

### Visualize Initial Fusion Result

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

SOURCE_COLORS = {
    'lidar':  '#4488ff',
    'camera': '#44ff88',
    'fused':  '#ff4444',
}
SOURCE_LABELS = {
    'lidar':  'LiDAR only',
    'camera': 'Camera only',
    'fused':  'Fused (both)',
}

fig, axes = plt.subplots(1, 2, figsize=(22, 9))
fig.patch.set_facecolor('#0a0a0a')

# ── Left: Camera image ────────────────────────────────────────────────────────
ax1 = axes[0]
img = np.array(Image.open(img_path))
ax1.imshow(img)
ax1.set_title('YOLO26n Detections — CAM_FRONT', color='white', fontsize=13)
ax1.axis('off')

for d in fused:
    if 'bbox_px' not in d or d['bbox_px'] is None:
        continue
    x1, y1, x2, y2 = d['bbox_px']
    color = SOURCE_COLORS[d['source']]
    rect  = patches.Rectangle(
        (x1, y1), x2-x1, y2-y1,
        linewidth=2.5, edgecolor=color, facecolor='none'
    )
    ax1.add_patch(rect)
    ax1.text(x1, y1-5,
             f"{d['class_name'][:3]} {d['score']:.2f}",
             color=color, fontsize=7, fontweight='bold')
    ax1.plot((x1+x2)/2, y2, 'o', color=color, markersize=4)

# ── Right: BEV fusion ─────────────────────────────────────────────────────────
ax2 = axes[1]
ax2.set_facecolor('#0a0a0a')
ax2.set_xlim(-35, 65)
ax2.set_ylim(-25, 20)
ax2.set_xlabel('X — forward (meters)', color='#aaaaaa', fontsize=11)
ax2.set_ylabel('Y — left (meters)',    color='#aaaaaa', fontsize=11)
ax2.set_title('Camera-LiDAR Late Fusion — Bird\'s Eye View\n'
              'Blue=LiDAR only | Green=Camera only | Red=Fused',
              color='white', fontsize=13, pad=12)
ax2.tick_params(colors='#888888')
for spine in ax2.spines.values():
    spine.set_edgecolor('#333333')
ax2.grid(True, color='#1a1a1a', linewidth=0.5)

# Range rings
for r in [10, 20, 30, 40, 50, 60]:
    circle = plt.Circle((0, 0), r, color='#2a2a2a',
                         fill=False, linewidth=0.5)
    ax2.add_patch(circle)
    ax2.text(r+0.3, 0.5, f'{r}m', color='#444444', fontsize=7)

# Camera FOV cone
fov_angle = np.radians(60)
fov_range = 65
for sign in [1, -1]:
    ax2.plot([0, fov_range * np.cos(sign * fov_angle)],
             [0, fov_range * np.sin(sign * fov_angle)],
             color='#333333', linewidth=0.8, linestyle='--')
ax2.text(52, 13, 'Camera\nFOV', color='#444444', fontsize=7)

# Plot detections
plotted_sources = set()
for d in fused:
    dist = float(np.linalg.norm(d['bev_xy']))
    if dist > 60:
        continue
    x, y   = d['bev_xy']
    source = d['source']
    color  = SOURCE_COLORS[source]
    size   = 100 + d['score'] * 80
    label  = SOURCE_LABELS[source] if source not in plotted_sources else None

    ax2.scatter(x, y, c=color, s=size, zorder=5,
                alpha=0.9, edgecolors='white',
                linewidths=0.5, label=label)
    plotted_sources.add(source)

    ax2.text(x+0.8, y+0.8,
             f"{d['class_name'][:3]}\n{d['score']:.2f}",
             color=color, fontsize=6, zorder=6, fontweight='bold')

    # Uncertainty circle for fused detections
    if source == 'fused':
        circle = plt.Circle((x, y), d.get('match_dist', 1.5),
                             color='#ff4444', fill=False,
                             linewidth=0.8, alpha=0.3, zorder=4)
        ax2.add_patch(circle)

# Ego vehicle
ego = patches.FancyBboxPatch(
    (-1, -1.5), 2, 3,
    boxstyle="round,pad=0.1",
    edgecolor='white', facecolor='#333333',
    linewidth=2, zorder=7
)
ax2.add_patch(ego)
ax2.text(0, 0, 'EGO', color='white', ha='center', va='center',
         fontsize=8, fontweight='bold', zorder=8)

ax2.annotate('', xy=(5, 0), xytext=(2.5, 0),
             arrowprops=dict(arrowstyle='->', color='#888888', lw=1.5))

# Legend
ax2.legend(loc='upper right', facecolor='#1a1a1a',
           labelcolor='white', fontsize=9,
           framealpha=0.9, edgecolor='#333333')

# Stats box
stats = (f"Camera dets : {len(camera_dets_filtered)}\n"
         f"LiDAR dets  : {len(real_lidar)}\n"
         f"Fused       : {n_fused}\n"
         f"LiDAR-only  : {n_lidar}\n"
         f"Camera-only : {n_camera}")
ax2.text(0.02, 0.98, stats,
         transform=ax2.transAxes, color='#aaaaaa', fontsize=8,
         verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='#1a1a1a',
                   alpha=0.8, edgecolor='#333333'))

plt.subplots_adjust(left=0.04, right=0.98, top=0.93, bottom=0.07)
plt.savefig('/content/fusion_bev_real.jpg', dpi=150,
            facecolor='#0a0a0a', pad_inches=0)
plt.show()
print("✅ Saved → /content/fusion_bev_real.jpg")

### Fix: Widen Match Threshold and Filter Front-Only LiDAR
Increase threshold from 3m to 8m (single-sweep LiDAR has position uncertainty) and drop LiDAR detections behind ego that cannot appear in the front camera.

In [ ]:
# Fix 1: Increase match threshold to account for single-sweep uncertainty
MATCH_DIST_THRESH = 8.0   # was 3.0m

# Fix 2: Only use LiDAR detections in front of ego (x > -5)
# Pedestrians behind ego can't match camera front view
real_lidar_front = [d for d in real_lidar
                    if d['bev_xy'][0] > -5]

print(f"LiDAR dets total      : {len(real_lidar)}")
print(f"LiDAR dets in front   : {len(real_lidar_front)}")

# Rerun fusion
camera_dets_filtered = [d for d in detections if d['distance'] <= 60.0]
fused = fuse_detections(camera_dets_filtered, real_lidar_front,
                        match_thresh=8.0)

n_fused  = sum(1 for d in fused if d['source'] == 'fused')
n_lidar  = sum(1 for d in fused if d['source'] == 'lidar')
n_camera = sum(1 for d in fused if d['source'] == 'camera')

print(f"\nFusion results (threshold=8m, front-only LiDAR):")
print(f"  Camera dets : {len(camera_dets_filtered)}")
print(f"  LiDAR dets  : {len(real_lidar_front)}")
print(f"  Fused       : {n_fused}")
print(f"  LiDAR-only  : {n_lidar}")
print(f"  Camera-only : {n_camera}")

print(f"\n  {'Source':<8} {'Class':<15} {'Score':>6} {'x':>8} {'y':>8}")
print(f"  {'-'*55}")
for d in sorted(fused,
                key=lambda x: float(np.linalg.norm(x['bev_xy']))):
    dist = float(np.linalg.norm(d['bev_xy']))
    if dist > 60:
        continue
    x, y = d['bev_xy']
    if d['source'] == 'fused':
        print(f"  {d['source']:<8} {d['class_name']:<15} "
              f"{d['score']:>6.3f} {x:>8.2f} {y:>8.2f}  ✅")
    else:
        print(f"  {d['source']:<8} {d['class_name']:<15} "
              f"{d['score']:>6.3f} {x:>8.2f} {y:>8.2f}")

### Visualize Improved Fusion

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

SOURCE_COLORS = {
    'lidar':  '#4488ff',
    'camera': '#44ff88',
    'fused':  '#ff4444',
}
SOURCE_LABELS = {
    'lidar':  'LiDAR only',
    'camera': 'Camera only',
    'fused':  'Fused (both)',
}

fig, axes = plt.subplots(1, 2, figsize=(22, 9))
fig.patch.set_facecolor('#0a0a0a')

# ── Left: Camera image ────────────────────────────────────────────────────────
ax1 = axes[0]
img = np.array(Image.open(img_path))
ax1.imshow(img)
ax1.set_title('YOLO26n Detections — CAM_FRONT\n'
              'Green=Camera only | Red=Fused with LiDAR',
              color='white', fontsize=12)
ax1.axis('off')

for d in fused:
    if 'bbox_px' not in d or d['bbox_px'] is None:
        continue
    x1, y1, x2, y2 = d['bbox_px']
    color = SOURCE_COLORS[d['source']]
    rect  = patches.Rectangle(
        (x1, y1), x2-x1, y2-y1,
        linewidth=2.5, edgecolor=color, facecolor='none'
    )
    ax1.add_patch(rect)
    ax1.text(x1, y1-5,
             f"{d['class_name'][:3]} {d['score']:.2f}",
             color=color, fontsize=7, fontweight='bold')
    ax1.plot((x1+x2)/2, y2, 'o', color=color, markersize=4)

# ── Right: BEV fusion ─────────────────────────────────────────────────────────
ax2 = axes[1]
ax2.set_facecolor('#0a0a0a')
ax2.set_xlim(-5, 65)
ax2.set_ylim(-20, 15)
ax2.set_xlabel('X — forward (meters)', color='#aaaaaa', fontsize=11)
ax2.set_ylabel('Y — left (meters)',    color='#aaaaaa', fontsize=11)
ax2.set_title('Camera-LiDAR Late Fusion — Bird\'s Eye View\n'
              'Blue=LiDAR only | Green=Camera only | Red=Fused',
              color='white', fontsize=13, pad=12)
ax2.tick_params(colors='#888888')
for spine in ax2.spines.values():
    spine.set_edgecolor('#333333')
ax2.grid(True, color='#1a1a1a', linewidth=0.5)

# Range rings
for r in [10, 20, 30, 40, 50, 60]:
    circle = plt.Circle((0, 0), r, color='#2a2a2a',
                         fill=False, linewidth=0.5)
    ax2.add_patch(circle)
    ax2.text(r+0.3, 0.3, f'{r}m', color='#444444', fontsize=7)

# Camera FOV cone
fov_angle = np.radians(60)
fov_range = 65
for sign in [1, -1]:
    ax2.plot([0, fov_range * np.cos(sign * fov_angle)],
             [0, fov_range * np.sin(sign * fov_angle)],
             color='#333333', linewidth=0.8, linestyle='--')
ax2.text(52, 12, 'Camera\nFOV', color='#444444', fontsize=7)

# Plot detections
plotted_sources = set()
for d in fused:
    dist = float(np.linalg.norm(d['bev_xy']))
    if dist > 60:
        continue
    x, y   = d['bev_xy']
    source = d['source']
    color  = SOURCE_COLORS[source]
    size   = 100 + d['score'] * 100
    label  = SOURCE_LABELS[source] if source not in plotted_sources else None

    ax2.scatter(x, y, c=color, s=size, zorder=5,
                alpha=0.9, edgecolors='white',
                linewidths=0.5, label=label)
    plotted_sources.add(source)

    ax2.text(x+0.5, y+0.5,
             f"{d['class_name'][:3]}\n{d['score']:.2f}",
             color=color, fontsize=6.5, zorder=6, fontweight='bold')

    # Uncertainty circle for fused
    if source == 'fused':
        circle = plt.Circle((x, y), 2.0,
                             color='#ff4444', fill=False,
                             linewidth=0.8, alpha=0.3, zorder=4)
        ax2.add_patch(circle)

# Ego vehicle
ego = patches.FancyBboxPatch(
    (-1, -1.5), 2, 3,
    boxstyle="round,pad=0.1",
    edgecolor='white', facecolor='#333333',
    linewidth=2, zorder=7
)
ax2.add_patch(ego)
ax2.text(0, 0, 'EGO', color='white', ha='center', va='center',
         fontsize=8, fontweight='bold', zorder=8)

ax2.annotate('', xy=(5, 0), xytext=(2.5, 0),
             arrowprops=dict(arrowstyle='->', color='#888888', lw=1.5))

# Legend
ax2.legend(loc='upper right', facecolor='#1a1a1a',
           labelcolor='white', fontsize=9,
           framealpha=0.9, edgecolor='#333333')

# Stats box
stats = (f"Camera dets : {len(camera_dets_filtered)}\n"
         f"LiDAR dets  : {len(real_lidar_front)}\n"
         f"Fused       : {n_fused}\n"
         f"LiDAR-only  : {n_lidar}\n"
         f"Camera-only : {n_camera}\n"
         f"Match thresh: 8.0m*\n"
         f"*single-sweep LiDAR")
ax2.text(0.02, 0.98, stats,
         transform=ax2.transAxes, color='#aaaaaa', fontsize=8,
         verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='#1a1a1a',
                   alpha=0.8, edgecolor='#333333'))

plt.subplots_adjust(left=0.04, right=0.98, top=0.93, bottom=0.07)
plt.savefig('/content/fusion_bev_real.jpg', dpi=150,
            facecolor='#0a0a0a', pad_inches=0)
plt.show()
print("✅ Saved → /content/fusion_bev_real.jpg")

### Debug: Distance Analysis — Why Some Camera Cars Did Not Fuse
Prints the BEV distance between every unmatched camera detection and all LiDAR detections to identify the gap.

In [ ]:
print("Distance check — why camera cars didn't fuse:\n")

# Camera cars that are camera-only
camera_cars = [(d['class_name'], d['bev_xy']) 
               for d in fused if d['source'] == 'camera' 
               and d['class_name'] in ['car','truck','bus','pedestrian']]

# All LiDAR detections (front only)
lidar_dets = [(d['class_name'], d['bev_xy'], d['score']) 
              for d in real_lidar_front]

for cls, cam_xy in camera_cars:
    print(f"Camera {cls} at ({cam_xy[0]:.1f}, {cam_xy[1]:.1f}):")
    for l_cls, l_xy, l_score in sorted(lidar_dets, 
                                        key=lambda x: np.linalg.norm(x[1]-cam_xy)):
        dist = float(np.linalg.norm(l_xy - cam_xy))
        if dist > 20:
            break
        matched = "✅ within 8m" if dist <= 8.0 else "❌ too far"
        print(f"  → LiDAR {l_cls} at ({l_xy[0]:.1f}, {l_xy[1]:.1f}) "
              f"dist={dist:.1f}m {matched}")
    print()

print("\nBus/Truck double detection issue:")
for d in fused:
    if d['class_name'] in ['bus', 'truck']:
        print(f"  {d['source']:<8} {d['class_name']:<10} "
              f"({d['bev_xy'][0]:.1f}, {d['bev_xy'][1]:.1f}) "
              f"score={d['score']:.3f}")

### LiDAR Deduplication and Fusion v2
PointPillars can fire duplicate detections on the same object with different class labels.  keeps only the highest-confidence detection per spatial cluster.

In [ ]:
def deduplicate_lidar(lidar_dets, dist_thresh=2.0):
    """
    Remove duplicate LiDAR detections at nearly identical positions.
    PointPillars sometimes detects the same object as multiple classes
    (e.g. bus + truck at same x,y). Keep highest confidence detection.
    """
    kept    = []
    used    = set()

    # Sort by score descending — keep best score first
    sorted_dets = sorted(lidar_dets,
                         key=lambda x: x['score'], reverse=True)

    for i, d in enumerate(sorted_dets):
        if i in used:
            continue
        kept.append(d)
        # Mark any nearby detection of similar class as duplicate
        for j, d2 in enumerate(sorted_dets):
            if j <= i or j in used:
                continue
            dist = float(np.linalg.norm(d['bev_xy'] - d2['bev_xy']))
            if dist < dist_thresh:
                used.add(j)
                print(f"  Dedup: removed {d2['class_name']} "
                      f"({d2['bev_xy'][0]:.1f},{d2['bev_xy'][1]:.1f}) "
                      f"score={d2['score']:.3f} — "
                      f"kept {d['class_name']} score={d['score']:.3f}")

    return kept


def fuse_detections_v2(camera_dets, lidar_dets,
                       match_thresh=8.0):
    """
    Improved late fusion with optimal matching.
    Instead of greedy per-LiDAR matching, build all valid pairs
    sorted by distance, then match closest pairs first.
    This prevents a far camera detection consuming a LiDAR detection
    that should match a closer camera detection.
    """
    # Build all valid candidate pairs
    candidates = []
    for li, ldet in enumerate(lidar_dets):
        for ci, cdet in enumerate(camera_dets):
            if not classes_compatible(ldet['class_name'],
                                      cdet['class_name']):
                continue
            dist = float(np.linalg.norm(
                ldet['bev_xy'] - cdet['bev_xy']
            ))
            if dist <= match_thresh:
                candidates.append((dist, li, ci))

    # Sort by distance — match closest pairs first
    candidates.sort(key=lambda x: x[0])

    matched_cam   = set()
    matched_lidar = set()
    fused         = []

    for dist, li, ci in candidates:
        if li in matched_lidar or ci in matched_cam:
            continue
        ldet = lidar_dets[li]
        cdet = camera_dets[ci]
        fused.append({
            'class_name':   ldet['class_name'],
            'score':        float(0.6*ldet['score'] + 0.4*cdet['score']),
            'bev_xy':       ldet['bev_xy'],
            'box_3d':       ldet.get('box_3d'),
            'bbox_px':      cdet.get('bbox_px'),
            'source':       'fused',
            'match_dist':   dist,
            'lidar_score':  ldet['score'],
            'camera_score': cdet['score'],
        })
        matched_lidar.add(li)
        matched_cam.add(ci)

    # Unmatched LiDAR
    for li, ldet in enumerate(lidar_dets):
        if li not in matched_lidar:
            d = dict(ldet)
            d['source'] = 'lidar'
            fused.append(d)

    # Unmatched camera
    for ci, cdet in enumerate(camera_dets):
        if ci not in matched_cam:
            d = dict(cdet)
            d['source'] = 'camera'
            fused.append(d)

    return fused


# Step 1: Deduplicate LiDAR
print("Deduplicating LiDAR detections:")
real_lidar_dedup = deduplicate_lidar(real_lidar_front, dist_thresh=2.0)
print(f"\nLiDAR before dedup: {len(real_lidar_front)}")
print(f"LiDAR after dedup : {len(real_lidar_dedup)}")

# Step 2: Run improved fusion
camera_dets_filtered = [d for d in detections if d['distance'] <= 60.0]
fused = fuse_detections_v2(camera_dets_filtered, real_lidar_dedup,
                            match_thresh=8.0)

n_fused  = sum(1 for d in fused if d['source'] == 'fused')
n_lidar  = sum(1 for d in fused if d['source'] == 'lidar')
n_camera = sum(1 for d in fused if d['source'] == 'camera')

print(f"\nImproved Fusion results:")
print(f"  Camera dets : {len(camera_dets_filtered)}")
print(f"  LiDAR dets  : {len(real_lidar_dedup)}")
print(f"  Fused       : {n_fused}")
print(f"  LiDAR-only  : {n_lidar}")
print(f"  Camera-only : {n_camera}")

print(f"\n  {'Source':<8} {'Class':<15} {'Score':>6} {'x':>8} {'y':>8}")
print(f"  {'-'*55}")
for d in sorted(fused,
                key=lambda x: float(np.linalg.norm(x['bev_xy']))):
    dist = float(np.linalg.norm(d['bev_xy']))
    if dist > 60:
        continue
    x, y = d['bev_xy']
    tag = " ✅" if d['source'] == 'fused' else ""
    print(f"  {d['source']:<8} {d['class_name']:<15} "
          f"{d['score']:>6.3f} {x:>8.2f} {y:>8.2f}{tag}")

### Visualize Fusion After LiDAR Deduplication

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

SOURCE_COLORS = {
    'lidar':  '#4488ff',
    'camera': '#44ff88',
    'fused':  '#ff4444',
}
SOURCE_LABELS = {
    'lidar':  'LiDAR only',
    'camera': 'Camera only',
    'fused':  'Fused (both)',
}

fig, axes = plt.subplots(1, 2, figsize=(22, 9))
fig.patch.set_facecolor('#0a0a0a')

# ── Left: Camera image ────────────────────────────────────────────────────────
ax1 = axes[0]
img = np.array(Image.open(img_path))
ax1.imshow(img)
ax1.set_title('YOLO26n Detections — CAM_FRONT\n'
              'Green=Camera only | Red=Fused with LiDAR',
              color='white', fontsize=12)
ax1.axis('off')

for d in fused:
    if 'bbox_px' not in d or d['bbox_px'] is None:
        continue
    x1, y1, x2, y2 = d['bbox_px']
    color = SOURCE_COLORS[d['source']]
    rect  = patches.Rectangle(
        (x1, y1), x2-x1, y2-y1,
        linewidth=2.5, edgecolor=color, facecolor='none'
    )
    ax1.add_patch(rect)
    ax1.text(x1, y1-5,
             f"{d['class_name'][:3]} {d['score']:.2f}",
             color=color, fontsize=7, fontweight='bold')
    ax1.plot((x1+x2)/2, y2, 'o', color=color, markersize=4)

# ── Right: BEV fusion ─────────────────────────────────────────────────────────
ax2 = axes[1]
ax2.set_facecolor('#0a0a0a')
ax2.set_xlim(-5, 65)
ax2.set_ylim(-20, 12)
ax2.set_xlabel('X — forward (meters)', color='#aaaaaa', fontsize=11)
ax2.set_ylabel('Y — left (meters)',    color='#aaaaaa', fontsize=11)
ax2.set_title('Camera-LiDAR Late Fusion — Bird\'s Eye View\n'
              'Blue=LiDAR only | Green=Camera only | Red=Fused',
              color='white', fontsize=13, pad=12)
ax2.tick_params(colors='#888888')
for spine in ax2.spines.values():
    spine.set_edgecolor('#333333')
ax2.grid(True, color='#1a1a1a', linewidth=0.5)

# Range rings
for r in [10, 20, 30, 40, 50, 60]:
    circle = plt.Circle((0, 0), r, color='#2a2a2a',
                         fill=False, linewidth=0.5)
    ax2.add_patch(circle)
    ax2.text(r+0.3, 0.3, f'{r}m', color='#444444', fontsize=7)

# Camera FOV cone
fov_angle = np.radians(60)
fov_range = 65
for sign in [1, -1]:
    ax2.plot([0, fov_range * np.cos(sign * fov_angle)],
             [0, fov_range * np.sin(sign * fov_angle)],
             color='#333333', linewidth=0.8, linestyle='--')
ax2.text(52, 10, 'Camera\nFOV', color='#444444', fontsize=7)

# Plot detections
plotted_sources = set()
for d in fused:
    dist = float(np.linalg.norm(d['bev_xy']))
    if dist > 60:
        continue
    x, y   = d['bev_xy']
    source = d['source']
    color  = SOURCE_COLORS[source]
    size   = 100 + d['score'] * 120
    label  = SOURCE_LABELS[source] if source not in plotted_sources else None

    ax2.scatter(x, y, c=color, s=size, zorder=5,
                alpha=0.9, edgecolors='white',
                linewidths=0.5, label=label)
    plotted_sources.add(source)

    ax2.text(x+0.5, y+0.5,
             f"{d['class_name'][:3]}\n{d['score']:.2f}",
             color=color, fontsize=6.5, zorder=6, fontweight='bold')

    # Uncertainty circle for fused
    if source == 'fused':
        circle = plt.Circle((x, y), 2.0,
                             color='#ff4444', fill=False,
                             linewidth=0.8, alpha=0.3, zorder=4)
        ax2.add_patch(circle)

# Ego vehicle
ego = patches.FancyBboxPatch(
    (-1, -1.5), 2, 3,
    boxstyle="round,pad=0.1",
    edgecolor='white', facecolor='#333333',
    linewidth=2, zorder=7
)
ax2.add_patch(ego)
ax2.text(0, 0, 'EGO', color='white', ha='center', va='center',
         fontsize=8, fontweight='bold', zorder=8)

ax2.annotate('', xy=(5, 0), xytext=(2.5, 0),
             arrowprops=dict(arrowstyle='->', color='#888888', lw=1.5))

# Legend
ax2.legend(loc='upper right', facecolor='#1a1a1a',
           labelcolor='white', fontsize=9,
           framealpha=0.9, edgecolor='#333333')

# Stats box
stats = (f"Camera dets  : {len(camera_dets_filtered)}\n"
         f"LiDAR dets   : {len(real_lidar_dedup)}\n"
         f"──────────────\n"
         f"Fused        : {n_fused}\n"
         f"LiDAR-only   : {n_lidar}\n"
         f"Camera-only  : {n_camera}\n"
         f"──────────────\n"
         f"Match thresh : 8.0m\n"
         f"*single-sweep LiDAR\n"
         f" (no sweep fusion)")
ax2.text(0.02, 0.98, stats,
         transform=ax2.transAxes, color='#aaaaaa', fontsize=8,
         verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='#1a1a1a',
                   alpha=0.8, edgecolor='#333333'))

plt.subplots_adjust(left=0.04, right=0.98, top=0.93, bottom=0.07)
plt.savefig('/content/fusion_bev_final.jpg', dpi=150,
            facecolor='#0a0a0a', pad_inches=0)
plt.show()
print("✅ Saved → /content/fusion_bev_final.jpg")

### Debug: Inspect Specific Unmatched Pairs
Manually check the BEV distance between known camera-only and LiDAR-only detections to decide on a final threshold.

In [ ]:
# Check specific unmatched pairs
cam_085 = next(d for d in fused if d['source']=='camera' 
               and d['class_name']=='car' 
               and abs(d['bev_xy'][0]-47.23) < 1)
cam_062 = next(d for d in fused if d['source']=='camera' 
               and d['class_name']=='car' 
               and abs(d['bev_xy'][0]-56.28) < 1)
lid_047 = next(d for d in fused if d['source']=='lidar' 
               and d['class_name']=='car' 
               and abs(d['bev_xy'][0]-36.76) < 1)
lid_031 = next(d for d in fused if d['source']=='lidar' 
               and d['class_name']=='car' 
               and abs(d['bev_xy'][0]-32.24) < 1)

d1 = np.linalg.norm(cam_085['bev_xy'] - lid_047['bev_xy'])
d2 = np.linalg.norm(cam_062['bev_xy'] - lid_031['bev_xy'])

print(f"Camera car 0.85 at {cam_085['bev_xy']}")
print(f"LiDAR  car 0.47 at {lid_047['bev_xy']}")
print(f"Distance: {d1:.1f}m\n")

print(f"Camera car 0.62 at {cam_062['bev_xy']}")
print(f"LiDAR  car 0.31 at {lid_031['bev_xy']}")
print(f"Distance: {d2:.1f}m")

### Tune: Rerun Fusion at 12m Threshold

In [ ]:
# Rerun with 12m threshold to catch the 10.5m pair
fused = fuse_detections_v2(camera_dets_filtered, real_lidar_dedup,
                            match_thresh=12.0)

n_fused  = sum(1 for d in fused if d['source'] == 'fused')
n_lidar  = sum(1 for d in fused if d['source'] == 'lidar')
n_camera = sum(1 for d in fused if d['source'] == 'camera')

print(f"Fusion results (threshold=12m):")
print(f"  Fused      : {n_fused}")
print(f"  LiDAR-only : {n_lidar}")
print(f"  Camera-only: {n_camera}")

# Check if car 0.85 is now fused
for d in sorted(fused, key=lambda x: float(np.linalg.norm(x['bev_xy']))):
    if d['class_name'] == 'car':
        x, y = d['bev_xy']
        print(f"  {d['source']:<8} car {d['score']:.2f} "
              f"({x:.1f}, {y:.1f})")

### Show All Cars in Fusion Output

In [ ]:
# Show all cars in the fusion result
print("All car detections in fusion output:")
print(f"\n{'Source':<8} {'Score':>6} {'x':>8} {'y':>8} {'Dist':>8}")
print("-"*45)
for d in sorted(fused, key=lambda x: float(np.linalg.norm(x['bev_xy']))):
    if d['class_name'] not in ['car', 'truck', 'bus']:
        continue
    x, y = d['bev_xy']
    dist = float(np.linalg.norm(d['bev_xy']))
    print(f"{d['source']:<8} {d['score']:>6.3f} {x:>8.2f} {y:>8.2f} {dist:>7.1f}m")

print("\nReal LiDAR cars (before fusion):")
for d in real_lidar_dedup:
    if d['class_name'] in ['car', 'truck', 'bus']:
        x, y = d['bev_xy']
        print(f"  {d['class_name']:<10} {d['score']:.3f} ({x:.1f}, {y:.1f})")

### Class-Aware Fusion v3
 adds a class-mismatch penalty so same-class pairs are preferred over cross-class pairs even when the cross-class distance is slightly shorter.

In [ ]:
def fuse_detections_v3(camera_dets, lidar_dets, match_thresh=12.0):
    """
    Fusion with class-aware scoring.
    Prioritize same-class matches over different-class matches
    even if distance is slightly larger.
    """
    candidates = []
    for li, ldet in enumerate(lidar_dets):
        for ci, cdet in enumerate(camera_dets):
            if not classes_compatible(ldet['class_name'],
                                      cdet['class_name']):
                continue
            dist = float(np.linalg.norm(
                ldet['bev_xy'] - cdet['bev_xy']
            ))
            if dist > match_thresh:
                continue

            # Penalize cross-class matches (e.g. truck↔car)
            same_class = ldet['class_name'] == cdet['class_name']
            # Also group truck/bus as same
            both_large = (ldet['class_name'] in {'truck','bus','trailer'} and
                         cdet['class_name'] in {'truck','bus','trailer'})
            class_penalty = 0.0 if (same_class or both_large) else 5.0

            score = dist + class_penalty
            candidates.append((score, dist, li, ci))

    # Sort by penalized score
    candidates.sort(key=lambda x: x[0])

    matched_cam   = set()
    matched_lidar = set()
    fused         = []

    for score, dist, li, ci in candidates:
        if li in matched_lidar or ci in matched_cam:
            continue
        ldet = lidar_dets[li]
        cdet = camera_dets[ci]
        fused.append({
            'class_name':   ldet['class_name'],
            'score':        float(0.6*ldet['score'] + 0.4*cdet['score']),
            'bev_xy':       ldet['bev_xy'],
            'box_3d':       ldet.get('box_3d'),
            'bbox_px':      cdet.get('bbox_px'),
            'source':       'fused',
            'match_dist':   dist,
            'lidar_score':  ldet['score'],
            'camera_score': cdet['score'],
        })
        matched_lidar.add(li)
        matched_cam.add(ci)

    for li, ldet in enumerate(lidar_dets):
        if li not in matched_lidar:
            d = dict(ldet)
            d['source'] = 'lidar'
            fused.append(d)

    for ci, cdet in enumerate(camera_dets):
        if ci not in matched_cam:
            d = dict(cdet)
            d['source'] = 'camera'
            fused.append(d)

    return fused


# Rerun with class-aware fusion
fused = fuse_detections_v3(camera_dets_filtered, real_lidar_dedup,
                            match_thresh=12.0)

n_fused  = sum(1 for d in fused if d['source'] == 'fused')
n_lidar  = sum(1 for d in fused if d['source'] == 'lidar')
n_camera = sum(1 for d in fused if d['source'] == 'camera')

print("All car/truck/bus detections after class-aware fusion:")
print(f"\n{'Source':<8} {'Class':<10} {'Score':>6} {'x':>8} {'y':>8}")
print("-"*48)
for d in sorted(fused,
                key=lambda x: float(np.linalg.norm(x['bev_xy']))):
    if d['class_name'] not in ['car','truck','bus']:
        continue
    x, y = d['bev_xy']
    print(f"{d['source']:<8} {d['class_name']:<10} "
          f"{d['score']:>6.3f} {x:>8.2f} {y:>8.2f}")

print(f"\nFused: {n_fused}  LiDAR-only: {n_lidar}  Camera-only: {n_camera}")

### Final Fusion Statistics and BEV Visualization

In [ ]:
# Update n values
n_fused  = sum(1 for d in fused if d['source'] == 'fused')
n_lidar  = sum(1 for d in fused if d['source'] == 'lidar')
n_camera = sum(1 for d in fused if d['source'] == 'camera')

# Same visualization code — just update stats text
stats = (f"Camera dets  : {len(camera_dets_filtered)}\n"
         f"LiDAR dets   : {len(real_lidar_dedup)}\n"
         f"──────────────\n"
         f"Fused        : {n_fused}\n"
         f"LiDAR-only   : {n_lidar}\n"
         f"Camera-only  : {n_camera}\n"
         f"──────────────\n"
         f"Match thresh : 12.0m\n"
         f"Class penalty: +5m\n"
         f"*single-sweep LiDAR")

# Re-run full visualization cell with updated fused and stats

### Verify Specific Match Assignments

In [ ]:
lidar_031 = next(d for d in real_lidar_dedup 
                 if d['class_name']=='car' 
                 and abs(d['bev_xy'][0]-32.24) < 1)

print(f"LiDAR car 0.31 at {lidar_031['bev_xy']}")
print(f"\nDistance to ALL camera detections:")
for d in sorted(camera_dets_filtered,
                key=lambda x: np.linalg.norm(x['bev_xy'] - lidar_031['bev_xy'])):
    dist = float(np.linalg.norm(d['bev_xy'] - lidar_031['bev_xy']))
    source = next((f['source'] for f in fused 
                   if 'bbox_px' in f and f.get('bbox_px') is not None
                   and np.allclose(f['bev_xy'], d['bev_xy'], atol=1.0)), 'unknown')
    print(f"  {d['class_name']:<12} ({d['bev_xy'][0]:.1f}, {d['bev_xy'][1]:.1f}) "
          f"dist={dist:.1f}m  → {source}")
    if dist > 20:
        break

In [ ]:
# Find which LiDAR detection matched camera car at (38.2, -7.4)
cam_car = next(d for d in camera_dets_filtered
               if d['class_name'] == 'car'
               and abs(d['bev_xy'][0] - 38.2) < 1)

print(f"Camera car at {cam_car['bev_xy']}")
print(f"\nWhich LiDAR matched this camera car?")
for d in fused:
    if d['source'] == 'fused' and 'bbox_px' in d and d['bbox_px'] is not None:
        if np.allclose(d['bbox_px'], cam_car['bbox_px'], atol=1.0):
            print(f"  → Fused with LiDAR {d['class_name']} "
                  f"at ({d['bev_xy'][0]:.1f}, {d['bev_xy'][1]:.1f}) "
                  f"score={d['lidar_score']:.3f}")